# Egg segmentation and individual video tracking

This cleaned notebook retains baseline and unsuccessful experiments, the final
image method, the final video tracker, and an updated report/export cell.

**Kaggle:** enable Internet and a GPU. Set dataset/video paths in Configuration.
Run in order. The visual-review confirmation intentionally stops execution until
you inspect the reference audit and related-image groups. Set the two review
flags only after review, then continue.

The final image algorithm is multi-threshold region proposals + GrabCut +
development-selected area filtering. The final video detector is separately
calibrated full-frame color/shape/local contrast; image scores do not measure its
accuracy. Existing baseline evaluation was viewed before follow-up development:
the final evaluation must be described as follow-up, not a fresh untouched test.

Before submission, complete truthful REPORT_NOTES, upload the final tracking video
to Google Drive, set VIDEO_OUTPUT_LINK, and rerun the last cell. Download this
notebook separately alongside the generated report and output ZIP.

Original experiment outputs are cleared to avoid stale results. This cleaned
notebook has been syntax/dependency-checked, but not run end-to-end on Kaggle.


## 1. Install dependencies


In [1]:
# Enable Internet and select a GPU in Notebook Settings.
# Original experiments ran on Kaggle; this cleaned notebook requires a fresh run.
# Run cells in order. No accuracy is guaranteed without running and reviewing results.

!pip install -q "transformers==4.49.0" "opencv-python-headless>=4.10,<4.12" "scikit-image>=0.24,<1" "gdown>=5,<6"

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.26.0 requires huggingface-hub<2.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
gradio 6.26.0 requires pillow<13.0,>=8.0, but you have pillow 13.0.0 which is incompatible.
diffusers 0.40.0 requires huggingface-hub<2.0,>=1.23.0, but you have huggingface-hub 0.36.2 which is incompatible.


## 2. Configuration


In [2]:

from pathlib import Path

# Leave None to download the alternative dataset ZIP from the assignment.
# Alternatively, upload the extracted dataset to Kaggle and specify its root.
DATA_ROOT = None
# DATA_ROOT = Path("/kaggle/input/egg-dataset")

OUTPUT = Path("/kaggle/working/egg_results")

# Upload a legally obtained copy of the assigned video to Kaggle.
# Assigned video: https://www.youtube.com/watch?v=Y4JyV94y2_A
VIDEO_PATH = Path('/kaggle/input/datasets/aburvan/egg-vid/vidssave.com YTB_1791655562357 360P.mp4')
# VIDEO_PATH = Path("/kaggle/input/egg-video/video.mp4")

# Fill after uploading tracked.mp4 to Google Drive.
# Enable sharing: "Anyone with the link".
VIDEO_OUTPUT_LINK = ""

SEED = 42
PHASH_DISTANCE = 6
BOX_THRESHOLD = 0.30
TEXT_THRESHOLD = 0.25

# Normally False: missing detections should be corrected rather than silently accepted.
ALLOW_EMPTY_REFERENCE = False

# Manually group related images to reduce leakage.
# Keys are paths relative to DATA_ROOT.
SOURCE_GROUPS = {
    # "train/class/image1.jpg": "capture_session_1",
    # "valid/class/image2.jpg": "capture_session_1",
}

# Correct missing/incorrect Grounding DINO boxes.
# Coordinates: [x1, y1, x2, y2] in the EXIF-corrected original image.
# SAM still generates the segmentation from these boxes.
BOX_OVERRIDES = {
    # "train/class/image.jpg": [[35, 20, 240, 290]],
}

# After running the reference-generation cell, inspect reference_audit.pdf and leakage_groups.csv.
# Correct boxes/groups as needed, rerun reference generation, then set these True.
REFERENCE_REVIEWED = False
SOURCE_GROUPS_REVIEWED = False

REPORT_NOTES = {
    "reference_review": "",
    "source_group_review": "",
    "qualitative_analysis": "",
    "video_review": (
        "In the prior final full-frame tracking output, visual review found "
        "egg detections without background objects labeled as eggs. "
        "Recheck the new output and record identity consistency through occlusion. "
        "This is qualitative review, not measured video accuracy."
    ),
    "ai_disclosure": (
        "AI assisted with the initial implementation. "
        "Describe your actual review, modifications, experiments, and AI use here."
    ),
    "experiment_notes": {
        # "hsv_relaxed": "Your actual observations from the development images."
    },
}

## 3. Reference masks and audit


In [3]:

import csv
import gc
import hashlib
import importlib.metadata
import json
import platform
import re
import textwrap
import time
import zipfile
from dataclasses import asdict, dataclass, replace
from urllib.parse import urlparse

import cv2
import gdown
import matplotlib
import numpy as np
import torch

matplotlib.use("Agg")
import matplotlib.pyplot as plt

from matplotlib.backends.backend_pdf import PdfPages
from PIL import Image as PILImage, ImageOps
from scipy import ndimage
from scipy.optimize import linear_sum_assignment
from skimage.feature import peak_local_max
from skimage.segmentation import watershed
from transformers import (
    AutoModelForZeroShotObjectDetection,
    AutoProcessor,
    SamModel,
    SamProcessor,
)

OUTPUT = Path(OUTPUT)
OUTPUT.mkdir(parents=True, exist_ok=True)

IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp", ".tif", ".tiff"}
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DETECTOR_NAME = "IDEA-Research/grounding-dino-tiny"
SAM_NAME = "facebook/sam-vit-base"

np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

if not 0 <= PHASH_DISTANCE <= 63:
    raise ValueError("PHASH_DISTANCE must be between 0 and 63.")

if not 0 < BOX_THRESHOLD < 1 or not 0 < TEXT_THRESHOLD < 1:
    raise ValueError("Detection thresholds must be between 0 and 1.")


def write_json(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(
        json.dumps(value, indent=2, allow_nan=False),
        encoding="utf-8",
    )


def write_csv(path, rows):
    if not rows:
        raise ValueError("Cannot write an empty CSV table.")

    with Path(path).open("w", newline="", encoding="utf-8") as handle:
        writer = csv.DictWriter(handle, fieldnames=list(rows[0]))
        writer.writeheader()
        writer.writerows(rows)


def sha256_file(path):
    digest = hashlib.sha256()

    with Path(path).open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(block)

    return digest.hexdigest()


def object_digest(value):
    encoded = json.dumps(value, sort_keys=True, allow_nan=False).encode("utf-8")
    return hashlib.sha256(encoded).hexdigest()


def read_rgb(path):
    with PILImage.open(path) as image:
        return np.asarray(
            ImageOps.exif_transpose(image).convert("RGB")
        ).copy()


def save_mask(path, labels):
    labels = np.asarray(labels)

    if (
        labels.ndim != 2
        or labels.min() < 0
        or labels.max() > 65535
    ):
        raise ValueError("Instance labels must be 2D integers in [0, 65535].")

    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    PILImage.fromarray(labels.astype(np.uint16)).save(path)


def load_mask(path, shape):
    with PILImage.open(path) as image:
        labels = np.asarray(image).copy()

    if labels.ndim != 2 or labels.shape != tuple(shape):
        raise ValueError(f"Mask shape mismatch: {path}")

    return labels


def perceptual_hash(rgb):
    gray = cv2.cvtColor(rgb, cv2.COLOR_RGB2GRAY)
    small = cv2.resize(gray, (32, 32), interpolation=cv2.INTER_AREA)
    coefficients = cv2.dct(
        small.astype(np.float32)
    )[:8, :8].ravel()[1:]

    median = np.median(coefficients)

    return sum(
        int(value > median) << index
        for index, value in enumerate(coefficients)
    )


def report_text_page(pdf, title, paragraphs):
    figure = plt.figure(figsize=(8.27, 11.69))
    figure.text(
        0.07, 0.95, title,
        fontsize=16, weight="bold", va="top",
    )
    y = 0.90

    for paragraph in paragraphs:
        wrapped = "\n".join(
            textwrap.fill(line, width=91) if line else ""
            for line in str(paragraph).splitlines()
        )
        line_count = wrapped.count("\n") + 1

        if y - line_count * 0.016 < 0.06:
            plt.close(figure)
            raise ValueError(
                f"Report page '{title}' overflows. Shorten custom notes."
            )

        figure.text(
            0.07, y, wrapped,
            fontsize=9, family="monospace", va="top",
        )
        y -= line_count * 0.016 + 0.02

    pdf.savefig(figure)
    plt.close(figure)


# ---------- Dataset ----------
if DATA_ROOT is None:
    DATA_ROOT = Path("/kaggle/input/datasets/aburvan/egg-classfication")
    DATA_ROOT.mkdir(parents=True, exist_ok=True)

    existing_images = [
        path for path in DATA_ROOT.rglob("*")
        if path.is_file() and path.suffix.lower() in IMAGE_EXTENSIONS
    ]

    if not existing_images:
        archive_path = Path("/kaggle/working/egg_dataset.zip")

        downloaded = gdown.download(
            id="1a58rACcCFnaj0ZiDoNyGbDWXunRyWziv",
            output=str(archive_path),
            quiet=False,
        )

        if downloaded is None or not zipfile.is_zipfile(archive_path):
            raise RuntimeError(
                "Dataset download failed. Upload the extracted dataset "
                "to Kaggle and set DATA_ROOT."
            )

        with zipfile.ZipFile(archive_path) as archive:
            for member in archive.infolist():
                destination = (DATA_ROOT / member.filename).resolve()

                if not destination.is_relative_to(DATA_ROOT.resolve()):
                    raise ValueError("Unsafe path inside dataset ZIP.")

            archive.extractall(DATA_ROOT)

DATA_ROOT = Path(DATA_ROOT).resolve()

if OUTPUT.resolve().is_relative_to(DATA_ROOT):
    raise ValueError("OUTPUT must be outside DATA_ROOT.")

paths = sorted(
    path for path in DATA_ROOT.rglob("*")
    if path.is_file() and path.suffix.lower() in IMAGE_EXTENSIONS
)

if len(paths) < 5:
    raise ValueError("At least five images are needed for a 4:1 split.")

records = []
hashes = []
source_names = []

for path in paths:
    relative = path.relative_to(DATA_ROOT).as_posix()
    identifier = hashlib.sha256(relative.encode()).hexdigest()[:20]
    rgb = read_rgb(path)

    records.append({
        "id": identifier,
        "path": relative,
        "shape": list(rgb.shape[:2]),
        "sha256": sha256_file(path),
    })

    hashes.append([
        perceptual_hash(rgb),
        perceptual_hash(np.fliplr(rgb)),
        perceptual_hash(np.flipud(rgb)),
    ])

    source_names.append(
        re.sub(r"\.rf\.[^.]+$", "", path.stem, flags=re.IGNORECASE)
    )

known_paths = {record["path"] for record in records}

for mapping in (SOURCE_GROUPS, BOX_OVERRIDES):
    unknown = set(mapping) - known_paths

    if unknown:
        raise ValueError(
            f"Configuration contains unknown image paths: {sorted(unknown)}"
        )


# ---------- Leakage-aware split ----------
parents = list(range(len(records)))


def find_root(index):
    while parents[index] != index:
        parents[index] = parents[parents[index]]
        index = parents[index]

    return index


for i in range(len(records)):
    for j in range(i):
        manual_match = (
            records[i]["path"] in SOURCE_GROUPS
            and records[j]["path"] in SOURCE_GROUPS
            and SOURCE_GROUPS[records[i]["path"]]
            == SOURCE_GROUPS[records[j]["path"]]
        )

        hash_distance = min(
            (a ^ b).bit_count()
            for a in hashes[i]
            for b in hashes[j]
        )

        if (
            manual_match
            or source_names[i] == source_names[j]
            or records[i]["sha256"] == records[j]["sha256"]
            or hash_distance <= PHASH_DISTANCE
        ):
            parents[find_root(i)] = find_root(j)

group_dictionary = {}

for index in range(len(records)):
    group_dictionary.setdefault(find_root(index), []).append(index)

groups = list(group_dictionary.values())
np.random.default_rng(SEED).shuffle(groups)

# Nearest 20% image count without breaking source/duplicate groups.
reachable = {0: []}

for group_index, group in enumerate(groups):
    for count, selected in list(reachable.items()):
        new_count = count + len(group)

        if new_count not in reachable:
            reachable[new_count] = selected + [group_index]

eligible_counts = [
    count for count in reachable
    if 0 < count < len(records)
]

if not eligible_counts:
    raise ValueError(
        "All images belong to one leakage group. Review SOURCE_GROUPS "
        "and PHASH_DISTANCE before proceeding."
    )

target_eval_count = len(records) / 5
eval_count = min(
    eligible_counts,
    key=lambda count: (abs(count - target_eval_count), count),
)
eval_group_indices = set(reachable[eval_count])

for group_index, group in enumerate(groups):
    for member in group:
        records[member]["group"] = group_index
        records[member]["split"] = (
            "eval" if group_index in eval_group_indices else "dev"
        )

dev_records = [
    record for record in records if record["split"] == "dev"
]
eval_records = [
    record for record in records if record["split"] == "eval"
]

split_manifest = {
    "seed": SEED,
    "phash_distance": PHASH_DISTANCE,
    "dev_images": len(dev_records),
    "eval_images": len(eval_records),
    "group_count": len(groups),
    "actual_eval_fraction": len(eval_records) / len(records),
    "images": records,
}

write_json(OUTPUT / "split.json", split_manifest)

write_csv(
    OUTPUT / "leakage_groups.csv",
    [
        {
            "path": record["path"],
            "group": record["group"],
            "split": record["split"],
        }
        for record in records
    ],
)

print(f"Dataset: {len(records)} images")
print(f"Split: {len(dev_records)} dev / {len(eval_records)} eval")

if eval_count != target_eval_count:
    print(
        "WARNING: Exact 4:1 is incompatible with the detected group sizes; "
        "the nearest group-preserving split is used."
    )


# ---------- Pretrained reference generation ----------
# Function-local inference tensors are released when the function returns.
# This also avoids undefined-variable cleanup when every image uses manual boxes.
def generate_reference_masks():
    print(f"Loading Grounding DINO and SAM on {DEVICE}...")

    detector_processor = AutoProcessor.from_pretrained(DETECTOR_NAME)
    detector = AutoModelForZeroShotObjectDetection.from_pretrained(
        DETECTOR_NAME
    ).to(DEVICE).eval()

    sam_processor = SamProcessor.from_pretrained(SAM_NAME)
    sam = SamModel.from_pretrained(SAM_NAME).to(DEVICE).eval()

    metadata = []

    try:
        for number, record in enumerate(records, 1):
            rgb = read_rgb(DATA_ROOT / record["path"])
            image = PILImage.fromarray(rgb)
            height, width = rgb.shape[:2]
            manual_boxes = record["path"] in BOX_OVERRIDES

            if manual_boxes:
                boxes = np.asarray(
                    BOX_OVERRIDES[record["path"]],
                    dtype=np.float32,
                ).reshape(-1, 4)

                detector_scores = [None] * len(boxes)

            else:
                detector_inputs = detector_processor(
                    images=image,
                    text="an egg.",
                    return_tensors="pt",
                ).to(DEVICE)

                with torch.inference_mode():
                    detector_outputs = detector(**detector_inputs)

                detections = (
                    detector_processor.post_process_grounded_object_detection(
                        detector_outputs,
                        detector_inputs.input_ids,
                        box_threshold=BOX_THRESHOLD,
                        text_threshold=TEXT_THRESHOLD,
                        target_sizes=[(height, width)],
                    )[0]
                )

                boxes = detections["boxes"].cpu().numpy()
                detector_scores = detections["scores"].cpu().tolist()

                del detector_inputs, detector_outputs, detections

            if len(boxes):
                if not np.isfinite(boxes).all():
                    raise ValueError(
                        f"Nonfinite box: {record['path']}"
                    )

                boxes[:, [0, 2]] = boxes[:, [0, 2]].clip(0, width)
                boxes[:, [1, 3]] = boxes[:, [1, 3]].clip(0, height)

                if np.any(boxes[:, 2:] <= boxes[:, :2]):
                    raise ValueError(
                        f"Invalid/outside-image box: {record['path']}"
                    )

            if not len(boxes) and not ALLOW_EMPTY_REFERENCE:
                raise RuntimeError(
                    f"No egg detected in: {record['path']}. "
                    "Add correct boxes to BOX_OVERRIDES and rerun the reference-generation cell. "
                    "Do not accept an empty reference unless the image "
                    "really contains no egg."
                )

            labels = np.zeros((height, width), dtype=np.int32)
            qualities = []

            if len(boxes):
                sam_inputs = sam_processor(
                    images=image,
                    input_boxes=[boxes.tolist()],
                    return_tensors="pt",
                ).to(DEVICE)

                with torch.inference_mode():
                    sam_outputs = sam(
                        **sam_inputs,
                        multimask_output=True,
                    )

                candidate_masks = (
                    sam_processor.image_processor.post_process_masks(
                        sam_outputs.pred_masks.cpu(),
                        sam_inputs["original_sizes"].cpu(),
                        sam_inputs["reshaped_input_sizes"].cpu(),
                    )[0].numpy()
                )

                predicted_ious = (
                    sam_outputs.iou_scores[0].cpu().numpy()
                )
                best_candidates = predicted_ious.argmax(axis=1)

                qualities = [
                    float(predicted_ious[index, candidate])
                    for index, candidate in enumerate(best_candidates)
                ]

                next_label = 1

                for box_index in np.argsort(qualities)[::-1]:
                    mask = candidate_masks[
                        box_index,
                        best_candidates[box_index],
                    ].astype(bool)

                    if not mask.any():
                        raise RuntimeError(
                            f"SAM produced an empty mask: {record['path']}"
                        )

                    overlap = (
                        np.count_nonzero(mask & (labels > 0))
                        / int(mask.sum())
                    )

                    if overlap > 0.8:
                        continue

                    labels[mask & (labels == 0)] = next_label
                    next_label += 1

                del sam_inputs, sam_outputs, candidate_masks, predicted_ious

            save_mask(
                OUTPUT / "reference_masks" / f"{record['id']}.png",
                labels,
            )

            warnings = []

            if not len(boxes):
                warnings.append(
                    "Explicitly allowed empty reference; inspect manually."
                )

            if any(value < 0.8 for value in qualities):
                warnings.append(
                    "Low SAM predicted mask quality; inspect manually."
                )

            metadata.append({
                **record,
                "boxes_xyxy": boxes.tolist(),
                "detector_scores": detector_scores,
                "sam_predicted_iou": qualities,
                "manual_boxes": manual_boxes,
                "instances": int(labels.max()),
                "warnings": warnings,
            })

            print(
                f"[{number}/{len(records)}] {record['path']}: "
                f"{int(labels.max())} egg(s)"
                + (f" — {warnings}" if warnings else "")
            )

    finally:
        del detector, sam
        gc.collect()

        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    return metadata


reference_metadata = generate_reference_masks()

write_json(
    OUTPUT / "reference_metadata.json",
    {
        "detector": DETECTOR_NAME,
        "sam": SAM_NAME,
        "box_threshold": BOX_THRESHOLD,
        "text_threshold": TEXT_THRESHOLD,
        "images": reference_metadata,
        "note": "SAM pseudo-ground-truth, not human ground truth.",
    },
)

reference_provenance = {
    "split_digest": object_digest(split_manifest),
    "metadata_digest": object_digest(reference_metadata),
    "mask_sha256": {
        record["id"]: sha256_file(
            OUTPUT / "reference_masks" / f"{record['id']}.png"
        )
        for record in records
    },
}

with PdfPages(OUTPUT / "reference_audit.pdf") as pdf:
    report_text_page(
        pdf,
        "Reference-mask review",
        [
            "Inspect every image for missed eggs, false detections, "
            "wrong boundaries, duplicated eggs, and merged instances.",
            "If corrections are needed, update BOX_OVERRIDES and rerun the reference-generation cell. "
            "The boxes prompt SAM; they are not segmentation masks.",
            "Inspect leakage_groups.csv and manually group images from "
            "the same source or capture session through SOURCE_GROUPS.",
            "SAM predicted IoU is confidence, not measured ground-truth IoU.",
        ],
    )

    for record, metadata in zip(records, reference_metadata):
        rgb = read_rgb(DATA_ROOT / record["path"])
        labels = load_mask(
            OUTPUT / "reference_masks" / f"{record['id']}.png",
            record["shape"],
        )

        figure, axes = plt.subplots(1, 2, figsize=(12, 5))
        axes[0].imshow(rgb)
        axes[0].set_title("Original")

        axes[1].imshow(rgb)
        axes[1].imshow(
            np.ma.masked_where(labels == 0, labels),
            cmap="nipy_spectral",
            alpha=0.55,
            interpolation="nearest",
        )
        axes[1].set_title("SAM reference instances")

        for axis in axes:
            axis.axis("off")

        figure.suptitle(
            f"{record['path']}\n"
            f"SAM predicted IoU: {metadata['sam_predicted_iou']}\n"
            f"Warnings: {metadata['warnings']}",
            fontsize=9,
        )
        figure.tight_layout()
        pdf.savefig(figure)
        plt.close(figure)

print("\nReview these files before running the classical experiments:")
print(OUTPUT / "reference_audit.pdf")
print(OUTPUT / "leakage_groups.csv")
print("After review, set REFERENCE_REVIEWED and SOURCE_GROUPS_REVIEWED to True.")

Dataset: 110 images
Split: 88 dev / 22 eval
Loading Grounding DINO and SAM on cuda...


preprocessor_config.json:   0%|          | 0.00/457 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/82.0 [00:00<?, ?B/s]

... (4 outputs left out)

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/375M [00:00<?, ?B/s]

/tmp/ipykernel_49/2210297398.py:414: FutureWarning: `box_threshold` is deprecated and will be removed in version 4.51.0 for `GroundingDinoProcessor.post_process_grounded_object_detection`. Use `threshold` instead.
  detector_processor.post_process_grounded_object_detection(
/usr/local/lib/python3.13/dist-packages/transformers/image_processing_utils.py:42: UserWarning: The following named arguments are not valid for `SamImageProcessor.preprocess` and were ignored: 'point_pad_value'
  return self.preprocess(images, **kwargs)


[1/110] test/egg/not_damaged_101_jpg.rf.479d12a2b41a5ec6d718dcdabd14b0c9.jpg: 1 egg(s)
[2/110] test/egg/not_damaged_125_jpg.rf.b19353b1f84dcbef1ec088b850927bb3.jpg: 1 egg(s)
[3/110] test/egg/not_damaged_14_jpg.rf.dc96559078b03433d2b65e5055904af1.jpg: 1 egg(s)
[4/110] test/egg/not_damaged_152_jpg.rf.a57edb18d395ce87e23f7abccbd1ad69.jpg: 1 egg(s)
[5/110] test/egg/not_damaged_153_jpg.rf.ea842e70f08e63e69bbcc74f1bc143a8.jpg: 1 egg(s)
[6/110] test/egg/not_damaged_161_jpg.rf.fd216aa5e11e147fc1527478cceae94e.jpg: 1 egg(s)
[7/110] test/egg/not_damaged_1_jpg.rf.d74c98a14f068b48c1f58fd573361f1f.jpg: 1 egg(s)
[8/110] test/egg/not_damaged_30_jpg.rf.48e9261777355fe67139ac1c5eae596b.jpg: 1 egg(s)
[9/110] test/egg/not_damaged_43_jpg.rf.66c31342150e751c30214786e72c8683.jpg: 1 egg(s)
[10/110] test/egg/not_damaged_4_jpg.rf.36c78b63ea4710d9642651ac3292fd58.jpg: 1 egg(s)
[11/110] test/egg/not_damaged_78_jpg.rf.d3a6e8624d192abc58ff7df9d5781bb5.jpg: 1 egg(s)
[12/110] train/Unlabeled/not_damaged_135_jpg.rf.a

## 4. Split and reference inspection


In [4]:
from collections import defaultdict
import matplotlib.pyplot as plt
import numpy as np

# ---------- Split checks ----------
group_members = defaultdict(list)

for record in records:
    group_members[record["group"]].append(record)

crossing_groups = {
    group_id: members
    for group_id, members in group_members.items()
    if len({member["split"] for member in members}) > 1
}

assert not crossing_groups, (
    f"Leakage groups cross dev/eval: {list(crossing_groups)}"
)

dev_count = sum(record["split"] == "dev" for record in records)
eval_count = sum(record["split"] == "eval" for record in records)

print(f"Images: {len(records)}")
print(f"Development: {dev_count}")
print(f"Evaluation: {eval_count}")
print(f"Groups: {len(group_members)}")
print(f"Exact 4:1 ratio: {dev_count == 4 * eval_count}")
print("No detected group crosses dev/eval.")

# ---------- Display helper ----------
def show_reference_samples(sample_records, title, columns=3):
    if not sample_records:
        print(f"{title}: none")
        return

    for offset in range(0, len(sample_records), 9):
        batch = sample_records[offset:offset + 9]
        rows = (len(batch) + columns - 1) // columns

        figure, axes = plt.subplots(
            rows, columns,
            figsize=(5 * columns, 4.5 * rows),
            squeeze=False,
        )
        axes = axes.ravel()

        for axis, record in zip(axes, batch):
            rgb = read_rgb(DATA_ROOT / record["path"])
            labels = load_mask(
                OUTPUT / "reference_masks" / f"{record['id']}.png",
                record["shape"],
            )

            axis.imshow(rgb)
            axis.imshow(
                np.ma.masked_where(labels == 0, labels),
                cmap="nipy_spectral",
                alpha=0.45,
                interpolation="nearest",
            )
            axis.set_title(
                f"{record['path']}\n"
                f"group={record['group']} | {record['split']} | "
                f"instances={len(np.unique(labels[labels > 0]))}",
                fontsize=8,
            )
            axis.axis("off")

        for axis in axes[len(batch):]:
            axis.axis("off")

        figure.suptitle(title, fontsize=14)
        figure.tight_layout()
        plt.show()
        plt.close(figure)

# ---------- Detected related-image groups ----------
multi_image_groups = {
    group_id: members
    for group_id, members in group_members.items()
    if len(members) > 1
}

print("\nDetected multi-image groups:")
for group_id, members in multi_image_groups.items():
    print(
        f"Group {group_id}:",
        [member["path"] for member in members],
    )
    show_reference_samples(members, f"Detected related group {group_id}")

# ---------- Multiple-instance reference masks ----------
metadata_by_id = {
    item["id"]: item for item in reference_metadata
}

multiple_instance_records = [
    record for record in records
    if metadata_by_id[record["id"]]["instances"] > 1
]

print(
    f"\nImages with multiple SAM instances: "
    f"{len(multiple_instance_records)}"
)
show_reference_samples(
    multiple_instance_records,
    "Check whether these multiple egg instances are genuine",
)

Images: 110
Development: 88
Evaluation: 22
Groups: 109
Exact 4:1 ratio: True
No detected group crosses dev/eval.

Detected multi-image groups:
Group 38: ['test/egg/not_damaged_78_jpg.rf.d3a6e8624d192abc58ff7df9d5781bb5.jpg', 'train/egg/not_damaged_27_jpg.rf.9eabbf76d6ac2d4caccb38b11e0d28ad.jpg']

Images with multiple SAM instances: 19


## 5. Confirm completed visual review


In [6]:
# Set True only after reviewing the current reference masks and source groups.
REFERENCE_REVIEWED = True
SOURCE_GROUPS_REVIEWED = True

print("Reference/source review explicitly confirmed.")

Reference/source review explicitly confirmed.


In [7]:
# Do not set these flags automatically without inspecting the references.
# If needed, edit SOURCE_GROUPS/BOX_OVERRIDES and regenerate the references.
if not REFERENCE_REVIEWED or not SOURCE_GROUPS_REVIEWED:
    raise RuntimeError(
        "Inspect reference_audit.pdf and leakage_groups.csv. "
        "Then set REFERENCE_REVIEWED=True and SOURCE_GROUPS_REVIEWED=True "
        "in this notebook session, complete REPORT_NOTES, and rerun this cell."
    )
print("Reference/source review explicitly confirmed.")


Reference/source review explicitly confirmed.


## 6. Initial classical experiments and baseline evaluation


In [8]:
# Run only after reviewing the files generated by Cell 3.

if not REFERENCE_REVIEWED or not SOURCE_GROUPS_REVIEWED:
    raise RuntimeError(
        "Inspect reference_audit.pdf and leakage_groups.csv first. "
        "Correct BOX_OVERRIDES/SOURCE_GROUPS if needed and rerun reference generation. "
        "Then set REFERENCE_REVIEWED=True and SOURCE_GROUPS_REVIEWED=True."
    )


def current_provenance():
    return {
        "split_digest": object_digest(split_manifest),
        "metadata_digest": object_digest(reference_metadata),
        "mask_sha256": {
            record["id"]: sha256_file(
                OUTPUT / "reference_masks" / f"{record['id']}.png"
            )
            for record in records
        },
    }


def verify_inputs():
    current_paths = {
        path.relative_to(DATA_ROOT).as_posix()
        for path in DATA_ROOT.rglob("*")
        if path.is_file() and path.suffix.lower() in IMAGE_EXTENSIONS
    }

    if current_paths != {record["path"] for record in records}:
        raise RuntimeError("Dataset image list changed. Rerun reference generation.")

    for record in records:
        if sha256_file(DATA_ROOT / record["path"]) != record["sha256"]:
            raise RuntimeError(
                f"Dataset image changed: {record['path']}. Rerun reference generation."
            )

    if current_provenance() != reference_provenance:
        raise RuntimeError("Reference masks changed. Rerun reference generation.")


verify_inputs()


@dataclass(frozen=True)
class ClassicalConfig:
    method: str = "hsv"
    max_side: int = 768
    saturation: float = 0.40
    value: float = 0.45
    contrast: float = 22.0
    kernel: int = 5
    min_area: float = 0.001
    max_area: float = 0.80
    min_solidity: float = 0.65
    min_circularity: float = 0.25
    max_axis_ratio: float = 3.5
    peak_fraction: float = 0.025
    split_touching: bool = True

    def __post_init__(self):
        if self.method not in {"otsu", "hsv", "border", "kmeans"}:
            raise ValueError(f"Unknown method: {self.method}")

        if self.max_side < 32 or self.kernel < 1 or self.kernel % 2 == 0:
            raise ValueError("Use max_side >= 32 and a positive odd kernel.")

        if not 0 < self.min_area < self.max_area <= 1:
            raise ValueError("Invalid area fractions.")

        if not 0 <= self.saturation <= 1 or not 0 <= self.value <= 1:
            raise ValueError("HSV thresholds must be in [0, 1].")

        if not 0 < self.peak_fraction < 0.5:
            raise ValueError("Invalid peak_fraction.")


def classical_segment(rgb, config):
    if rgb.ndim != 3 or rgb.shape[2] != 3 or rgb.dtype != np.uint8:
        raise ValueError("Expected a uint8 RGB image.")

    height, width = rgb.shape[:2]
    scale = min(1.0, config.max_side / max(height, width))

    if scale < 1:
        work = cv2.resize(
            rgb,
            (
                max(1, round(width * scale)),
                max(1, round(height * scale)),
            ),
            interpolation=cv2.INTER_AREA,
        )
    else:
        work = rgb.copy()

    work = cv2.GaussianBlur(work, (5, 5), 0)
    hsv = cv2.cvtColor(work, cv2.COLOR_RGB2HSV).astype(np.float32)
    lab = cv2.cvtColor(work, cv2.COLOR_RGB2LAB).astype(np.float32)

    pale = (
        (hsv[..., 1] <= config.saturation * 255)
        & (hsv[..., 2] >= config.value * 255)
    )

    border = np.zeros(work.shape[:2], dtype=bool)
    band = max(1, round(min(border.shape) * 0.04))
    border[:band] = border[-band:] = True
    border[:, :band] = border[:, -band:] = True

    if config.method == "otsu":
        gray = cv2.cvtColor(work, cv2.COLOR_RGB2GRAY)
        _, binary = cv2.threshold(
            gray, 0, 255,
            cv2.THRESH_BINARY + cv2.THRESH_OTSU,
        )
        mask = binary > 0

    elif config.method == "hsv":
        mask = pale

    elif config.method == "border":
        samples = lab[border][::max(1, int(border.sum()) // 2000)]
        cv2.setRNGSeed(SEED)

        _, _, centers = cv2.kmeans(
            samples,
            min(4, len(samples)),
            None,
            (cv2.TERM_CRITERIA_MAX_ITER, 30, 0.1),
            3,
            cv2.KMEANS_PP_CENTERS,
        )

        distances = np.stack([
            np.linalg.norm(lab - center, axis=2)
            for center in centers
        ])
        mask = distances.min(axis=0) > config.contrast

    else:
        flat = lab.reshape(-1, 3)
        samples = flat[::max(1, len(flat) // 20000)]
        cv2.setRNGSeed(SEED)

        _, _, centers = cv2.kmeans(
            samples,
            min(5, len(samples)),
            None,
            (cv2.TERM_CRITERIA_MAX_ITER, 40, 0.1),
            3,
            cv2.KMEANS_PP_CENTERS,
        )

        distances = np.stack([
            np.linalg.norm(lab - center, axis=2)
            for center in centers
        ])
        clusters = distances.argmin(axis=0)
        mask = np.zeros(clusters.shape, dtype=bool)

        for index in range(len(centers)):
            region = clusters == index

            if not region.any():
                continue

            image_fraction = float(region.mean())
            border_fraction = float(region[border].mean())
            enrichment = image_fraction / max(
                border_fraction, 1 / int(border.sum())
            )
            plausible_color = float(pale[region].mean()) > 0.5

            if enrichment > 1.5 and (
                plausible_color or centers[index, 0] > 85
            ):
                mask |= region

    binary = mask.astype(np.uint8) * 255

    if config.kernel > 1:
        kernel = cv2.getStructuringElement(
            cv2.MORPH_ELLIPSE,
            (config.kernel, config.kernel),
        )
        binary = cv2.morphologyEx(binary, cv2.MORPH_OPEN, kernel)
        binary = cv2.morphologyEx(binary, cv2.MORPH_CLOSE, kernel)

    binary = ndimage.binary_fill_holes(binary > 0).astype(np.uint8)

    contours, _ = cv2.findContours(
        binary,
        cv2.RETR_EXTERNAL,
        cv2.CHAIN_APPROX_SIMPLE,
    )
    foreground = np.zeros(binary.shape, dtype=np.uint8)

    for contour in contours:
        area = cv2.contourArea(contour)
        perimeter = cv2.arcLength(contour, True)
        hull_area = cv2.contourArea(cv2.convexHull(contour))
        _, (axis_a, axis_b), _ = cv2.minAreaRect(contour)

        axis_ratio = max(axis_a, axis_b) / max(min(axis_a, axis_b), 1)
        circularity = 4 * np.pi * area / max(perimeter ** 2, 1)
        solidity = area / max(hull_area, 1)

        if (
            config.min_area <= area / binary.size <= config.max_area
            and solidity >= config.min_solidity
            and circularity >= config.min_circularity
            and axis_ratio <= config.max_axis_ratio
        ):
            cv2.drawContours(
                foreground, [contour], -1, 1, cv2.FILLED
            )

    if config.split_touching and foreground.any():
        distance = ndimage.distance_transform_edt(foreground)
        smooth_distance = ndimage.gaussian_filter(distance, sigma=1)
        separation = max(
            3, round(min(foreground.shape) * config.peak_fraction)
        )

        peaks = peak_local_max(
            smooth_distance,
            min_distance=separation,
            threshold_abs=1,
            labels=foreground,
            exclude_border=False,
        )

        markers = np.zeros(foreground.shape, dtype=np.int32)

        for marker_id, (y, x) in enumerate(peaks, 1):
            markers[y, x] = marker_id

        components, component_count = ndimage.label(foreground)
        next_marker = len(peaks) + 1

        for component_id in range(1, component_count + 1):
            region = components == component_id

            if not markers[region].any():
                y, x = np.unravel_index(
                    np.argmax(distance * region),
                    distance.shape,
                )
                markers[y, x] = next_marker
                next_marker += 1

        instances = watershed(
            -smooth_distance,
            markers,
            mask=foreground,
        )

    else:
        instances, _ = ndimage.label(foreground)

    cleaned = np.zeros(instances.shape, dtype=np.int32)
    next_label = 1

    for instance_id in np.unique(instances):
        if instance_id and np.count_nonzero(
            instances == instance_id
        ) >= config.min_area * instances.size:
            cleaned[instances == instance_id] = next_label
            next_label += 1

    if scale < 1:
        cleaned = cv2.resize(
            cleaned,
            (width, height),
            interpolation=cv2.INTER_NEAREST,
        )

    return cleaned


def segmentation_metrics(truth_instances, predicted_instances):
    if (
        truth_instances.ndim != 2
        or truth_instances.shape != predicted_instances.shape
    ):
        raise ValueError("Metrics require matching 2D instance masks.")

    truth = truth_instances > 0
    prediction = predicted_instances > 0

    tp = int(np.count_nonzero(truth & prediction))
    fp = int(np.count_nonzero(~truth & prediction))
    fn = int(np.count_nonzero(truth & ~prediction))
    union = tp + fp + fn
    both_empty = union == 0

    true_edge = truth ^ ndimage.binary_erosion(truth)
    pred_edge = prediction ^ ndimage.binary_erosion(prediction)

    if not true_edge.any() or not pred_edge.any():
        boundary_f1 = float(
            not true_edge.any() and not pred_edge.any()
        )

    else:
        boundary_precision = float(np.mean(
            ndimage.distance_transform_edt(~true_edge)[pred_edge] <= 2
        ))
        boundary_recall = float(np.mean(
            ndimage.distance_transform_edt(~pred_edge)[true_edge] <= 2
        ))

        denominator = boundary_precision + boundary_recall
        boundary_f1 = (
            2 * boundary_precision * boundary_recall / denominator
            if denominator else 0.0
        )

    true_count = len(np.unique(truth_instances[truth]))
    pred_count = len(np.unique(predicted_instances[prediction]))

    return {
        "iou": tp / union if union else 1.0,
        "dice": 2 * tp / (2 * tp + fp + fn) if union else 1.0,
        "precision": tp / (tp + fp) if tp + fp else float(both_empty),
        "recall": tp / (tp + fn) if tp + fn else 1.0,
        "boundary_f1": boundary_f1,
        "tp": tp,
        "fp": fp,
        "fn": fn,
        "truth_instances": true_count,
        "pred_instances": pred_count,
        "count_error": abs(true_count - pred_count),
        "truth_fraction": float(truth.mean()),
    }


def aggregate(rows):
    if not rows:
        raise ValueError("Cannot aggregate an empty evaluation.")

    names = [
        "iou", "dice", "precision",
        "recall", "boundary_f1", "count_error",
    ]
    result = {
        name: float(np.mean([row[name] for row in rows]))
        for name in names
    }

    tp, fp, fn = (
        sum(row[name] for row in rows)
        for name in ["tp", "fp", "fn"]
    )
    result["micro_iou"] = (
        tp / (tp + fp + fn) if tp + fp + fn else 1.0
    )
    result["images"] = len(rows)

    return result


def score_record(record, config):
    rgb = read_rgb(DATA_ROOT / record["path"])
    truth = load_mask(
        OUTPUT / "reference_masks" / f"{record['id']}.png",
        record["shape"],
    )

    started = time.perf_counter()
    prediction = classical_segment(rgb, config)
    elapsed = time.perf_counter() - started

    return {
        "id": record["id"],
        "path": record["path"],
        "group": record["group"],
        "seconds": elapsed,
        **segmentation_metrics(truth, prediction),
    }, prediction


# Sanity checks run here; they do not establish real-dataset accuracy.
empty = np.zeros((32, 32), dtype=np.int32)
square = empty.copy()
square[8:24, 8:24] = 1

assert segmentation_metrics(empty, empty)["iou"] == 1.0
assert segmentation_metrics(square, square)["dice"] == 1.0
assert segmentation_metrics(square, empty)["iou"] == 0.0
assert segmentation_metrics(empty, square)["boundary_f1"] == 0.0

synthetic = np.zeros((160, 200, 3), dtype=np.uint8)
cv2.ellipse(
    synthetic, (100, 80), (30, 45),
    0, 0, 360, (230, 230, 230), -1,
)
synthetic_prediction = classical_segment(
    synthetic, ClassicalConfig(split_touching=False)
)
assert synthetic_prediction.shape == synthetic.shape[:2]
assert synthetic_prediction.max() == 1


base = ClassicalConfig()
trial_definitions = [
    (
        "otsu",
        replace(base, method="otsu", kernel=1, split_touching=False),
        "Brightness-only baseline.",
    ),
    (
        "hsv",
        replace(base, kernel=1, split_touching=False),
        "Reject saturated backgrounds using a pale-shell color prior.",
    ),
    (
        "hsv_cleanup",
        replace(base, split_touching=False),
        "Opening/closing removes noise and fills small gaps.",
    ),
    (
        "hsv_watershed",
        base,
        "Distance watershed separates touching foreground objects.",
    ),
    (
        "hsv_relaxed",
        replace(base, saturation=0.60, value=0.30),
        "Recover dim/brown eggs; may increase skin/background leakage.",
    ),
    (
        "hsv_strict",
        replace(base, saturation=0.25, value=0.60),
        "Increase white-egg precision; may omit shadows/brown shells.",
    ),
    (
        "border",
        replace(base, method="border"),
        "Use Lab novelty against clustered border-background colors.",
    ),
    (
        "border_relaxed",
        replace(base, method="border", contrast=14),
        "Recover weaker contrast; may admit more background.",
    ),
    (
        "kmeans",
        replace(base, method="kmeans"),
        "Select foreground-enriched Lab clusters.",
    ),
    (
        "kmeans_small",
        replace(base, method="kmeans", min_area=0.0003),
        "Retain smaller eggs; may retain small background objects.",
    ),
    (
        "hsv_fine_peaks",
        replace(base, peak_fraction=0.015),
        "Separate smaller touching eggs; may over-split.",
    ),
    (
        "hsv_coarse_peaks",
        replace(base, peak_fraction=0.05),
        "Reduce over-splitting; may merge neighboring eggs.",
    ),
]

experiments = []

for trial_name, config, rationale in trial_definitions:
    rows = [
        score_record(record, config)[0]
        for record in dev_records
    ]
    summary = aggregate(rows)

    experiments.append({
        "name": trial_name,
        "config": asdict(config),
        "rationale": rationale,
        "dev": summary,
        "mean_seconds": float(np.mean([
            row["seconds"] for row in rows
        ])),
        "per_image": rows,
    })

    print(
        f"{trial_name:20s} "
        f"dev IoU={summary['iou']:.4f} "
        f"Dice={summary['dice']:.4f}"
    )

winner_index = max(
    range(len(experiments)),
    key=lambda index: (
        experiments[index]["dev"]["iou"],
        experiments[index]["dev"]["dice"],
        -index,
    ),
)

best_experiment = experiments[winner_index]
BEST_CONFIG = ClassicalConfig(**best_experiment["config"])

for index, experiment in enumerate(experiments):
    experiment["decision"] = (
        "Kept: highest development macro IoU; "
        "Dice then earlier trial order break ties."
        if index == winner_index else
        f"Discarded: development IoU {experiment['dev']['iou']:.4f}; "
        f"selected IoU {best_experiment['dev']['iou']:.4f}. "
        "Tie rules favor the selected trial."
    )

write_json(OUTPUT / "experiments.json", {
    "provenance": reference_provenance,
    "trials": experiments,
})
write_json(OUTPUT / "best_config.json", {
    "trial": best_experiment["name"],
    "config": asdict(BEST_CONFIG),
    "dev": best_experiment["dev"],
    "provenance": reference_provenance,
})

write_csv(
    OUTPUT / "experiments.csv",
    [
        {
            "trial": experiment["name"],
            "parameters": json.dumps(
                experiment["config"], sort_keys=True
            ),
            "rationale": experiment["rationale"],
            "dev_iou": experiment["dev"]["iou"],
            "dev_dice": experiment["dev"]["dice"],
            "dev_boundary_f1": experiment["dev"]["boundary_f1"],
            "dev_count_mae": experiment["dev"]["count_error"],
            "seconds_per_image": experiment["mean_seconds"],
            "decision": experiment["decision"],
        }
        for experiment in experiments
    ],
)

print(f"\nFrozen final configuration: {best_experiment['name']}")

evaluation_rows = []

for record in eval_records:
    scores, prediction = score_record(record, BEST_CONFIG)
    evaluation_rows.append(scores)

    save_mask(
        OUTPUT / "classical_eval_masks" / f"{record['id']}.png",
        prediction,
    )

evaluation_summary = aggregate(evaluation_rows)

group_scores = {}
for row in evaluation_rows:
    group_scores.setdefault(row["group"], []).append(row["iou"])

group_values = list(group_scores.values())
rng = np.random.default_rng(SEED)
bootstrap_means = []

for _ in range(2000):
    sampled = rng.integers(
        0, len(group_values), size=len(group_values)
    )
    bootstrap_means.append(float(np.mean([
        value
        for index in sampled
        for value in group_values[index]
    ])))

evaluation_summary["iou_group_bootstrap_95ci"] = [
    float(value)
    for value in np.percentile(bootstrap_means, [2.5, 97.5])
]
evaluation_summary["eval_groups"] = len(group_values)

evaluation_provenance = reference_provenance.copy()
evaluation_config_digest = object_digest(asdict(BEST_CONFIG))

write_json(OUTPUT / "evaluation.json", {
    "trial": best_experiment["name"],
    "config": asdict(BEST_CONFIG),
    "provenance": evaluation_provenance,
    "summary": evaluation_summary,
    "images": evaluation_rows,
})
write_csv(OUTPUT / "evaluation.csv", evaluation_rows)

print("\nHeld-out results:")
print(json.dumps(evaluation_summary, indent=2))

otsu                 dev IoU=0.3094 Dice=0.3721
hsv                  dev IoU=0.1927 Dice=0.2277
hsv_cleanup          dev IoU=0.2093 Dice=0.2502
hsv_watershed        dev IoU=0.2095 Dice=0.2503
hsv_relaxed          dev IoU=0.0904 Dice=0.1142
hsv_strict           dev IoU=0.3489 Dice=0.4176
border               dev IoU=0.3293 Dice=0.4154
border_relaxed       dev IoU=0.1905 Dice=0.2425
kmeans               dev IoU=0.2895 Dice=0.3433
kmeans_small         dev IoU=0.2865 Dice=0.3408
hsv_fine_peaks       dev IoU=0.2097 Dice=0.2505
hsv_coarse_peaks     dev IoU=0.2093 Dice=0.2502

Frozen final configuration: hsv_strict

Held-out results:
{
  "iou": 0.3365690530527247,
  "dice": 0.417549079443237,
  "precision": 0.40252822551594003,
  "recall": 0.4938857499162974,
  "boundary_f1": 0.2050237531437971,
  "count_error": 2.272727272727273,
  "micro_iou": 0.3473925204719467,
  "images": 22,
  "iou_group_bootstrap_95ci": [
    0.2093206181998923,
    0.47175758850227206
  ],
  "eval_groups": 22
}


## 7. Watershed diagnostic


In [9]:
# DEVELOPMENT-ONLY DIAGNOSTICS
# Does not change BEST_CONFIG, overwrite evaluation, or use eval for selection.

from dataclasses import replace
import matplotlib.pyplot as plt
import numpy as np

verify_inputs()

selected_config = ClassicalConfig(**best_experiment["config"])
no_watershed_config = replace(
    selected_config,
    split_touching=False,
)

diagnostic_rows = {}
diagnostic_predictions = {}

for name, config in [
    ("selected", selected_config),
    ("without_watershed", no_watershed_config),
]:
    rows = []
    predictions = {}

    for record in dev_records:
        scores, prediction = score_record(record, config)
        rows.append(scores)
        predictions[record["id"]] = prediction

    diagnostic_rows[name] = rows
    diagnostic_predictions[name] = predictions

    summary = aggregate(rows)
    signed_count_error = float(np.mean([
        row["pred_instances"] - row["truth_instances"]
        for row in rows
    ]))

    print(f"\n{name}")
    print(f"Development IoU:       {summary['iou']:.4f}")
    print(f"Development Dice:      {summary['dice']:.4f}")
    print(f"Development precision: {summary['precision']:.4f}")
    print(f"Development recall:    {summary['recall']:.4f}")
    print(f"Development count MAE: {summary['count_error']:.4f}")
    print(f"Mean predicted - reference count: {signed_count_error:+.3f}")
    print(
        "Empty predictions:",
        sum(row["pred_instances"] == 0 for row in rows),
        "/",
        len(rows),
    )

# Retain actual ablation results for the experiment log.
write_json(
    OUTPUT / "development_watershed_diagnostic.json",
    {
        "scope": "development only",
        "purpose": "Separate foreground errors from watershed/count errors.",
        "variants": {
            name: {
                "config": asdict(config),
                "summary": aggregate(diagnostic_rows[name]),
                "per_image": diagnostic_rows[name],
            }
            for name, config in [
                ("selected", selected_config),
                ("without_watershed", no_watershed_config),
            ]
        },
    },
)

record_by_id = {
    record["id"]: record
    for record in dev_records
}

worst_rows = sorted(
    diagnostic_rows["selected"],
    key=lambda row: row["iou"],
)[:8]

for rank, row in enumerate(worst_rows, 1):
    record = record_by_id[row["id"]]
    rgb = read_rgb(DATA_ROOT / record["path"])

    truth = load_mask(
        OUTPUT / "reference_masks" / f"{record['id']}.png",
        record["shape"],
    )
    prediction = diagnostic_predictions["selected"][record["id"]]
    no_watershed = diagnostic_predictions["without_watershed"][record["id"]]

    error_image = (rgb * 0.35).astype(np.uint8)
    error_image[(truth > 0) & (prediction > 0)] = (0, 200, 0)
    error_image[(truth == 0) & (prediction > 0)] = (255, 0, 0)
    error_image[(truth > 0) & (prediction == 0)] = (255, 0, 255)

    figure, axes = plt.subplots(1, 5, figsize=(20, 4))

    axes[0].imshow(rgb)
    axes[0].set_title("Original")

    axes[1].imshow(
        truth,
        cmap="nipy_spectral",
        interpolation="nearest",
    )
    axes[1].set_title(f"SAM: {row['truth_instances']} instances")

    axes[2].imshow(
        prediction,
        cmap="nipy_spectral",
        interpolation="nearest",
    )
    axes[2].set_title(f"Selected: {row['pred_instances']} instances")

    axes[3].imshow(
        no_watershed,
        cmap="nipy_spectral",
        interpolation="nearest",
    )
    axes[3].set_title("Without watershed")

    axes[4].imshow(error_image)
    axes[4].set_title("Green TP / red FP / magenta FN")

    for axis in axes:
        axis.axis("off")

    figure.suptitle(
        f"DEV failure {rank}: {record['path']}\n"
        f"IoU={row['iou']:.3f}, "
        f"precision={row['precision']:.3f}, "
        f"recall={row['recall']:.3f}",
        fontsize=10,
    )
    figure.tight_layout()

    figure.savefig(
        OUTPUT / f"dev_failure_{rank}.png",
        dpi=150,
        bbox_inches="tight",
    )

    plt.show()
    plt.close(figure)


selected
Development IoU:       0.3489
Development Dice:      0.4176
Development precision: 0.4649
Development recall:    0.4904
Development count MAE: 2.3068
Mean predicted - reference count: +1.966
Empty predictions: 12 / 88

without_watershed
Development IoU:       0.3487
Development Dice:      0.4175
Development precision: 0.4646
Development recall:    0.4904
Development count MAE: 1.4773
Mean predicted - reference count: +1.000
Empty predictions: 12 / 88


## 8. HSV stage diagnostic


In [10]:
# NEW CELL — Locate the failing stage on the three worst development images.
# Uses SAM only to measure stage retention, not to generate predictions.

config = ClassicalConfig(**best_experiment["config"])

worst_dev_rows = sorted(
    diagnostic_rows["selected"],
    key=lambda row: row["iou"],
)[:3]

dev_record_by_id = {
    record["id"]: record
    for record in dev_records
}

for row in worst_dev_rows:
    record = dev_record_by_id[row["id"]]
    rgb = read_rgb(DATA_ROOT / record["path"])

    reference = load_mask(
        OUTPUT / "reference_masks" / f"{record['id']}.png",
        record["shape"],
    ) > 0

    height, width = rgb.shape[:2]
    scale = min(1.0, config.max_side / max(height, width))

    if scale < 1:
        work = cv2.resize(
            rgb,
            (
                max(1, round(width * scale)),
                max(1, round(height * scale)),
            ),
            interpolation=cv2.INTER_AREA,
        )
    else:
        work = rgb.copy()

    blurred = cv2.GaussianBlur(work, (5, 5), 0)
    hsv = cv2.cvtColor(
        blurred,
        cv2.COLOR_RGB2HSV,
    ).astype(np.float32)

    raw_mask = (
        (hsv[..., 1] <= config.saturation * 255)
        & (hsv[..., 2] >= config.value * 255)
    )

    cleaned = raw_mask.astype(np.uint8) * 255

    if config.kernel > 1:
        kernel = cv2.getStructuringElement(
            cv2.MORPH_ELLIPSE,
            (config.kernel, config.kernel),
        )
        cleaned = cv2.morphologyEx(
            cleaned, cv2.MORPH_OPEN, kernel
        )
        cleaned = cv2.morphologyEx(
            cleaned, cv2.MORPH_CLOSE, kernel
        )

    cleaned = ndimage.binary_fill_holes(cleaned > 0)

    final_instances = classical_segment(rgb, config)
    final_mask = final_instances > 0

    def original_size(mask):
        return cv2.resize(
            mask.astype(np.uint8),
            (width, height),
            interpolation=cv2.INTER_NEAREST,
        ) > 0

    raw_original = original_size(raw_mask)
    cleaned_original = original_size(cleaned)

    stages = [
        ("HSV threshold", raw_original),
        ("After morphology + fill", cleaned_original),
        ("After filtering + splitting", final_mask),
    ]

    print("\nIMAGE:", record["path"])

    for stage_name, mask in stages:
        scores = segmentation_metrics(
            reference.astype(np.int32),
            mask.astype(np.int32),
        )

        print(
            f"{stage_name:29s} "
            f"egg-pixel recall={scores['recall']:.3f} | "
            f"precision={scores['precision']:.3f} | "
            f"image foreground={mask.mean():.3f}"
        )

    figure, axes = plt.subplots(1, 5, figsize=(20, 4))

    axes[0].imshow(rgb)
    axes[0].set_title("Original")

    axes[1].imshow(reference, cmap="gray", vmin=0, vmax=1)
    axes[1].set_title("SAM reference")

    axes[2].imshow(raw_original, cmap="gray", vmin=0, vmax=1)
    axes[2].set_title("Raw HSV threshold")

    axes[3].imshow(cleaned_original, cmap="gray", vmin=0, vmax=1)
    axes[3].set_title("Morphology + hole filling")

    axes[4].imshow(final_mask, cmap="gray", vmin=0, vmax=1)
    axes[4].set_title("Final prediction")

    for axis in axes:
        axis.axis("off")

    figure.suptitle(record["path"], fontsize=9)
    figure.tight_layout()

    figure.savefig(
        OUTPUT / f"stage_diagnostic_{record['id']}.png",
        dpi=150,
        bbox_inches="tight",
    )

    plt.show()
    plt.close(figure)
    


IMAGE: test/egg/not_damaged_125_jpg.rf.b19353b1f84dcbef1ec088b850927bb3.jpg
HSV threshold                 egg-pixel recall=0.995 | precision=0.045 | image foreground=0.775
After morphology + fill       egg-pixel recall=0.995 | precision=0.044 | image foreground=0.784
After filtering + splitting   egg-pixel recall=0.000 | precision=0.000 | image foreground=0.000

IMAGE: test/egg/not_damaged_152_jpg.rf.a57edb18d395ce87e23f7abccbd1ad69.jpg
HSV threshold                 egg-pixel recall=0.906 | precision=0.122 | image foreground=0.399
After morphology + fill       egg-pixel recall=0.906 | precision=0.120 | image foreground=0.406
After filtering + splitting   egg-pixel recall=0.000 | precision=0.000 | image foreground=0.003

IMAGE: test/egg/not_damaged_78_jpg.rf.d3a6e8624d192abc58ff7df9d5781bb5.jpg
HSV threshold                 egg-pixel recall=0.626 | precision=0.205 | image foreground=0.430
After morphology + fill       egg-pixel recall=0.769 | precision=0.234 | image foreground=0.464
Af

## 9. Recorded baseline findings


In [11]:
REPORT_NOTES["experiment_notes"]["hsv_strict"] = (
    "Development-stage inspection showed that HSV retained 99.5%, 90.6%, "
    "and 62.6% of reference egg pixels in three zero-IoU examples. "
    "After morphology/hole filling, recall was 99.5%, 90.6%, and 76.9%, "
    "but precision remained only 4.4%, 12.0%, and 23.4%. "
    "Final component filtering removed all retained egg pixels. "
    "This indicates insufficient foreground/background separation before "
    "whole-component rejection, rather than a watershed-only failure."
)

## 10. Ellipse/GrabCut experiments and helpers


In [12]:
# NEW EXPERIMENT — Classical ellipse proposals + local GrabCut
# Uses development images only.
# Requires the earlier metrics and reference cells.

from dataclasses import dataclass, asdict
import time

import cv2
import numpy as np
import matplotlib.pyplot as plt

verify_inputs()


@dataclass(frozen=True)
class EllipseGrabCutConfig:
    max_side: int = 640
    canny_low: int = 35
    canny_high: int = 100
    edge_tolerance: float = 3.0
    min_edge_support: float = 0.35
    min_area_fraction: float = 0.003
    max_area_fraction: float = 0.35
    max_axis_ratio: float = 2.5
    max_candidates: int = 6
    grabcut_iterations: int = 4
    roi_expansion: float = 1.45


def binary_iou(a, b):
    intersection = np.count_nonzero(a & b)
    union = np.count_nonzero(a | b)
    return intersection / union if union else 1.0


def ellipse_mask(shape, ellipse, factor=1.0):
    (cx, cy), (axis_a, axis_b), angle = ellipse

    scaled = (
        (float(cx), float(cy)),
        (float(axis_a * factor), float(axis_b * factor)),
        float(angle),
    )

    mask = np.zeros(shape, dtype=np.uint8)
    cv2.ellipse(mask, scaled, 1, -1)
    return mask > 0


def find_ellipse_proposals(rgb, config):
    gray = cv2.cvtColor(rgb, cv2.COLOR_RGB2GRAY)
    gray = cv2.GaussianBlur(gray, (5, 5), 0)

    edges = cv2.Canny(
        gray,
        config.canny_low,
        config.canny_high,
    )

    # Connect short contour gaps without using a foreground color threshold.
    linked_edges = cv2.morphologyEx(
        edges,
        cv2.MORPH_CLOSE,
        cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3)),
    )

    if not edges.any():
        return []

    distance_to_edge = cv2.distanceTransform(
        (edges == 0).astype(np.uint8),
        cv2.DIST_L2,
        3,
    )

    contours, _ = cv2.findContours(
        linked_edges,
        cv2.RETR_LIST,
        cv2.CHAIN_APPROX_NONE,
    )

    height, width = gray.shape
    proposals = []

    for contour in contours:
        if len(contour) < 20:
            continue

        # Reject near-linear contours before fitting an ellipse.
        _, (rect_a, rect_b), _ = cv2.minAreaRect(contour)
        if min(rect_a, rect_b) < 4:
            continue

        ellipse = cv2.fitEllipse(contour)
        (cx, cy), (axis_a, axis_b), angle = ellipse

        values = np.array(
            [cx, cy, axis_a, axis_b, angle],
            dtype=float,
        )

        if not np.isfinite(values).all():
            continue

        if min(axis_a, axis_b) < 10:
            continue

        axis_ratio = max(axis_a, axis_b) / min(axis_a, axis_b)
        area_fraction = (
            np.pi * axis_a * axis_b / 4
        ) / (height * width)

        if not (
            config.min_area_fraction
            <= area_fraction
            <= config.max_area_fraction
        ):
            continue

        if axis_ratio > config.max_axis_ratio:
            continue

        # This candidate method intentionally requires an interior ellipse.
        # Border-clipped eggs are a known limitation to document.
        extent = max(axis_a, axis_b) / 2
        if (
            cx - extent < 1
            or cy - extent < 1
            or cx + extent >= width - 1
            or cy + extent >= height - 1
        ):
            continue

        outline = np.zeros(gray.shape, dtype=np.uint8)
        cv2.ellipse(outline, ellipse, 1, 1)
        outline_pixels = outline > 0

        support = float(np.mean(
            distance_to_edge[outline_pixels]
            <= config.edge_tolerance
        ))

        if support < config.min_edge_support:
            continue

        mask = ellipse_mask(gray.shape, ellipse)

        proposals.append({
            "ellipse": ellipse,
            "mask": mask,
            "edge_support": support,
        })

    # Suppress multiple fitted contours describing the same region.
    proposals.sort(
        key=lambda item: item["edge_support"],
        reverse=True,
    )

    selected = []

    for proposal in proposals:
        if any(
            binary_iou(proposal["mask"], old["mask"]) > 0.5
            for old in selected
        ):
            continue

        selected.append(proposal)

        if len(selected) >= config.max_candidates:
            break

    return selected


def refine_proposal(rgb, proposal, config):
    ellipse = proposal["ellipse"]
    (cx, cy), (axis_a, axis_b), angle = ellipse
    height, width = rgb.shape[:2]

    radius = max(axis_a, axis_b) * config.roi_expansion / 2

    x1 = max(0, int(np.floor(cx - radius)))
    y1 = max(0, int(np.floor(cy - radius)))
    x2 = min(width, int(np.ceil(cx + radius)))
    y2 = min(height, int(np.ceil(cy + radius)))

    crop = rgb[y1:y2, x1:x2]

    if min(crop.shape[:2]) < 10:
        return None

    local_ellipse = (
        (float(cx - x1), float(cy - y1)),
        (float(axis_a), float(axis_b)),
        float(angle),
    )

    original = ellipse_mask(crop.shape[:2], local_ellipse)
    inner = ellipse_mask(crop.shape[:2], local_ellipse, factor=0.35)
    outer = ellipse_mask(crop.shape[:2], local_ellipse, factor=1.20)

    initialization = np.full(
        crop.shape[:2],
        cv2.GC_PR_BGD,
        dtype=np.uint8,
    )

    initialization[~outer] = cv2.GC_BGD
    initialization[original] = cv2.GC_PR_FGD
    initialization[inner] = cv2.GC_FGD

    # Ensure background seeds exist at the crop boundary.
    initialization[0, :] = cv2.GC_BGD
    initialization[-1, :] = cv2.GC_BGD
    initialization[:, 0] = cv2.GC_BGD
    initialization[:, -1] = cv2.GC_BGD

    if not np.any(initialization == cv2.GC_FGD):
        return None

    background_model = np.zeros((1, 65), dtype=np.float64)
    foreground_model = np.zeros((1, 65), dtype=np.float64)

    cv2.setRNGSeed(SEED)
    cv2.grabCut(
        cv2.cvtColor(crop, cv2.COLOR_RGB2BGR),
        initialization,
        None,
        background_model,
        foreground_model,
        config.grabcut_iterations,
        cv2.GC_INIT_WITH_MASK,
    )

    foreground = (
        (initialization == cv2.GC_FGD)
        | (initialization == cv2.GC_PR_FGD)
    )

    # Retain the connected component containing the proposal's center.
    components, _ = ndimage.label(foreground)
    center_x = int(np.clip(round(cx - x1), 0, crop.shape[1] - 1))
    center_y = int(np.clip(round(cy - y1), 0, crop.shape[0] - 1))
    component_id = components[center_y, center_x]

    if component_id == 0:
        return None

    foreground = ndimage.binary_fill_holes(
        components == component_id
    )

    # Reject refinements that collapse or expand far beyond their proposal.
    area_ratio = (
        np.count_nonzero(foreground)
        / max(np.count_nonzero(original), 1)
    )

    if not 0.30 <= area_ratio <= 1.80:
        return None

    contours, _ = cv2.findContours(
        foreground.astype(np.uint8),
        cv2.RETR_EXTERNAL,
        cv2.CHAIN_APPROX_SIMPLE,
    )

    if not contours:
        return None

    contour = max(contours, key=cv2.contourArea)
    area = cv2.contourArea(contour)
    hull_area = cv2.contourArea(cv2.convexHull(contour))
    perimeter = cv2.arcLength(contour, True)

    solidity = area / max(hull_area, 1)
    circularity = 4 * np.pi * area / max(perimeter ** 2, 1)

    if solidity < 0.75 or circularity < 0.35:
        return None

    full_mask = np.zeros((height, width), dtype=bool)
    full_mask[y1:y2, x1:x2] = foreground

    return full_mask


def ellipse_grabcut_segment(rgb, config):
    height, width = rgb.shape[:2]
    scale = min(1.0, config.max_side / max(height, width))

    if scale < 1:
        work = cv2.resize(
            rgb,
            (
                max(1, round(width * scale)),
                max(1, round(height * scale)),
            ),
            interpolation=cv2.INTER_AREA,
        )
    else:
        work = rgb.copy()

    proposals = find_ellipse_proposals(work, config)
    accepted_masks = []

    for proposal in proposals:
        refined = refine_proposal(work, proposal, config)

        if refined is None:
            continue

        if any(
            binary_iou(refined, previous) > 0.5
            for previous in accepted_masks
        ):
            continue

        accepted_masks.append(refined)

    labels = np.zeros(work.shape[:2], dtype=np.int32)
    next_id = 1

    for mask in accepted_masks:
        available = mask & (labels == 0)

        # Do not create fragmented labels from overlapping proposals.
        components, count = ndimage.label(available)
        if count == 0:
            continue

        sizes = np.bincount(components.ravel())
        sizes[0] = 0
        largest = components == int(sizes.argmax())

        if np.count_nonzero(largest) < 0.5 * np.count_nonzero(mask):
            continue

        labels[largest] = next_id
        next_id += 1

    if scale < 1:
        labels = cv2.resize(
            labels,
            (width, height),
            interpolation=cv2.INTER_NEAREST,
        )

    return labels


# Declare the experiments before examining their results.
ellipse_trials = [
    (
        "ellipse_gc_default",
        EllipseGrabCutConfig(),
        "Edge-supported ellipse proposals followed by local GrabCut.",
    ),
    (
        "ellipse_gc_low_edges",
        EllipseGrabCutConfig(
            canny_low=20,
            canny_high=65,
        ),
        "Recover weaker shell boundaries; may admit background contours.",
    ),
    (
        "ellipse_gc_strict_support",
        EllipseGrabCutConfig(
            min_edge_support=0.50,
        ),
        "Reject weak ellipse proposals; may miss partly obscured eggs.",
    ),
    (
        "ellipse_gc_wider_roi",
        EllipseGrabCutConfig(
            roi_expansion=1.80,
        ),
        "Provide more local background samples for GrabCut.",
    ),
]

new_experiments = []
new_predictions = {}

for trial_name, trial_config, rationale in ellipse_trials:
    rows = []
    predictions = {}

    print(f"\nRunning {trial_name}...")

    for index, record in enumerate(dev_records, 1):
        rgb = read_rgb(DATA_ROOT / record["path"])
        truth = load_mask(
            OUTPUT / "reference_masks" / f"{record['id']}.png",
            record["shape"],
        )

        started = time.perf_counter()
        prediction = ellipse_grabcut_segment(rgb, trial_config)
        elapsed = time.perf_counter() - started

        predictions[record["id"]] = prediction

        rows.append({
            "id": record["id"],
            "path": record["path"],
            "group": record["group"],
            "seconds": elapsed,
            **segmentation_metrics(truth, prediction),
        })

        if index % 20 == 0:
            print(f"  {index}/{len(dev_records)} development images")

    summary = aggregate(rows)

    new_experiments.append({
        "name": trial_name,
        "config": asdict(trial_config),
        "rationale": rationale,
        "dev": summary,
        "mean_seconds": float(np.mean([
            row["seconds"] for row in rows
        ])),
        "empty_predictions": sum(
            row["pred_instances"] == 0 for row in rows
        ),
        "per_image": rows,
    })

    new_predictions[trial_name] = predictions

    print(
        f"IoU={summary['iou']:.4f} | "
        f"Dice={summary['dice']:.4f} | "
        f"Precision={summary['precision']:.4f} | "
        f"Recall={summary['recall']:.4f} | "
        f"Count MAE={summary['count_error']:.4f}"
    )


winner_index = max(
    range(len(new_experiments)),
    key=lambda index: (
        new_experiments[index]["dev"]["iou"],
        new_experiments[index]["dev"]["dice"],
        -index,
    ),
)

candidate_winner = new_experiments[winner_index]
baseline_iou = best_experiment["dev"]["iou"]
candidate_iou = candidate_winner["dev"]["iou"]

for index, experiment in enumerate(new_experiments):
    experiment["decision"] = (
        "Best new candidate by development IoU, then Dice and trial order."
        if index == winner_index else
        "Not selected among these new candidates."
    )

write_json(
    OUTPUT / "ellipse_grabcut_experiments.json",
    {
        "scope": "development only",
        "provenance": reference_provenance,
        "baseline_trial": best_experiment["name"],
        "baseline_dev": best_experiment["dev"],
        "trials": new_experiments,
        "note": (
            "These follow-up experiments were developed after viewing the "
            "initial baseline evaluation. No evaluation images were used "
            "in this cell. The original evaluation is not a fresh test set."
        ),
    },
)

write_csv(
    OUTPUT / "ellipse_grabcut_experiments.csv",
    [
        {
            "trial": experiment["name"],
            "parameters": json.dumps(
                experiment["config"], sort_keys=True
            ),
            "rationale": experiment["rationale"],
            "dev_iou": experiment["dev"]["iou"],
            "dev_dice": experiment["dev"]["dice"],
            "dev_precision": experiment["dev"]["precision"],
            "dev_recall": experiment["dev"]["recall"],
            "dev_count_mae": experiment["dev"]["count_error"],
            "empty_predictions": experiment["empty_predictions"],
            "seconds_per_image": experiment["mean_seconds"],
            "decision": experiment["decision"],
        }
        for experiment in new_experiments
    ],
)

print("\nCOMPARISON")
print(f"Original baseline dev IoU: {baseline_iou:.4f}")
print(
    f"Best new candidate: {candidate_winner['name']} "
    f"dev IoU={candidate_iou:.4f}"
)
print(f"Change in dev IoU: {candidate_iou - baseline_iou:+.4f}")
print("BEST_CONFIG and evaluation results were NOT changed.")

# Compare the same three previously diagnosed development failures.
baseline_rows = (
    diagnostic_rows["selected"]
    if "diagnostic_rows" in globals()
    else next(
        experiment["per_image"]
        for experiment in experiments
        if experiment["name"] == best_experiment["name"]
    )
)

selected_failures = sorted(
    baseline_rows,
    key=lambda row: row["iou"],
)[:3]

record_by_id = {
    record["id"]: record for record in dev_records
}
winner_predictions = new_predictions[candidate_winner["name"]]

for row in selected_failures:
    record = record_by_id[row["id"]]
    rgb = read_rgb(DATA_ROOT / record["path"])
    truth = load_mask(
        OUTPUT / "reference_masks" / f"{record['id']}.png",
        record["shape"],
    )
    prediction = winner_predictions[record["id"]]
    scores = segmentation_metrics(truth, prediction)

    figure, axes = plt.subplots(1, 3, figsize=(12, 4))

    axes[0].imshow(rgb)
    axes[0].set_title("Original")

    axes[1].imshow(
        truth,
        cmap="nipy_spectral",
        interpolation="nearest",
    )
    axes[1].set_title("SAM reference")

    axes[2].imshow(
        prediction,
        cmap="nipy_spectral",
        interpolation="nearest",
    )
    axes[2].set_title(
        f"New candidate\nIoU={scores['iou']:.3f}"
    )

    for axis in axes:
        axis.axis("off")

    figure.suptitle(record["path"], fontsize=9)
    figure.tight_layout()
    plt.show()
    plt.close(figure)


Running ellipse_gc_default...
  20/88 development images
  40/88 development images
  60/88 development images
  80/88 development images
IoU=0.3375 | Dice=0.3632 | Precision=0.3979 | Recall=0.3593 | Count MAE=1.2614

Running ellipse_gc_low_edges...
  20/88 development images
  40/88 development images
  60/88 development images
  80/88 development images
IoU=0.3449 | Dice=0.3718 | Precision=0.3872 | Recall=0.3764 | Count MAE=1.3295

Running ellipse_gc_strict_support...
  20/88 development images
  40/88 development images
  60/88 development images
  80/88 development images
IoU=0.2460 | Dice=0.2661 | Precision=0.3036 | Recall=0.2594 | Count MAE=1.3409

Running ellipse_gc_wider_roi...
  20/88 development images
  40/88 development images
  60/88 development images
  80/88 development images
IoU=0.3382 | Dice=0.3637 | Precision=0.3979 | Recall=0.3601 | Count MAE=1.2500

COMPARISON
Original baseline dev IoU: 0.3489
Best new candidate: ellipse_gc_low_edges dev IoU=0.3449
Change in dev I

## 11. Multi-threshold region/GrabCut experiment


In [13]:
# NEW DEVELOPMENT EXPERIMENT
# Multi-threshold region proposals -> existing local GrabCut refinement.
# No SAM masks/boxes are used to create predictions.

from dataclasses import asdict
import time
import cv2
import numpy as np

verify_inputs()

region_config = EllipseGrabCutConfig(
    max_candidates=12,
    min_edge_support=0.20,
    roi_expansion=1.60,
)


def region_proposals(rgb, config):
    height, width = rgb.shape[:2]

    blurred = cv2.GaussianBlur(rgb, (5, 5), 0)
    lab = cv2.cvtColor(blurred, cv2.COLOR_RGB2LAB)
    hsv = cv2.cvtColor(blurred, cv2.COLOR_RGB2HSV)

    # Different channels provide different foreground/background contrasts.
    channels = [
        lab[..., 0],                  # Lightness
        lab[..., 1],                  # Green/red contrast
        lab[..., 2],                  # Blue/yellow contrast
        hsv[..., 2],                  # Brightness
        255 - hsv[..., 1],            # Low saturation
    ]

    candidates = []
    kernel = cv2.getStructuringElement(
        cv2.MORPH_ELLIPSE, (3, 3)
    )

    for channel in channels:
        # Fixed, declared quantiles: not chosen from evaluation scores.
        thresholds = np.unique(
            np.percentile(channel, [30, 45, 60, 75, 85])
        )

        for threshold in thresholds:
            for bright_side in [True, False]:
                binary = (
                    channel > threshold
                    if bright_side
                    else channel <= threshold
                ).astype(np.uint8) * 255

                binary = cv2.morphologyEx(
                    binary, cv2.MORPH_OPEN, kernel
                )
                binary = cv2.morphologyEx(
                    binary, cv2.MORPH_CLOSE, kernel
                )

                contours, _ = cv2.findContours(
                    binary,
                    cv2.RETR_LIST,
                    cv2.CHAIN_APPROX_NONE,
                )

                for contour in contours:
                    if len(contour) < 20:
                        continue

                    area = cv2.contourArea(contour)
                    fraction = area / (height * width)

                    if not (
                        config.min_area_fraction
                        <= fraction
                        <= config.max_area_fraction
                    ):
                        continue

                    x, y, w, h = cv2.boundingRect(contour)

                    # Border-connected background is not an egg proposal.
                    # Border-clipped eggs remain a limitation of this trial.
                    if (
                        x <= 1 or y <= 1
                        or x + w >= width - 1
                        or y + h >= height - 1
                    ):
                        continue

                    _, (rect_a, rect_b), _ = cv2.minAreaRect(contour)
                    if min(rect_a, rect_b) < 8:
                        continue

                    ellipse = cv2.fitEllipse(contour)
                    (cx, cy), (axis_a, axis_b), angle = ellipse

                    if not np.isfinite(
                        [cx, cy, axis_a, axis_b, angle]
                    ).all():
                        continue

                    if min(axis_a, axis_b) < 10:
                        continue

                    if (
                        max(axis_a, axis_b) / min(axis_a, axis_b)
                        > config.max_axis_ratio
                    ):
                        continue

                    # Reject fitted ellipses extending outside the image.
                    theta = np.deg2rad(angle)
                    radius_a, radius_b = axis_a / 2, axis_b / 2
                    extent_x = np.sqrt(
                        (radius_a * np.cos(theta)) ** 2
                        + (radius_b * np.sin(theta)) ** 2
                    )
                    extent_y = np.sqrt(
                        (radius_a * np.sin(theta)) ** 2
                        + (radius_b * np.cos(theta)) ** 2
                    )

                    if (
                        cx - extent_x < 1
                        or cy - extent_y < 1
                        or cx + extent_x >= width - 1
                        or cy + extent_y >= height - 1
                    ):
                        continue

                    region = np.zeros((height, width), dtype=np.uint8)
                    cv2.drawContours(
                        region, [contour], -1, 1, cv2.FILLED
                    )
                    region = region > 0

                    fitted_mask = ellipse_mask(
                        (height, width), ellipse
                    )
                    agreement = binary_iou(region, fitted_mask)

                    if agreement < 0.65:
                        continue

                    # Shape agreement ranks proposals; size only breaks ties.
                    candidates.append({
                        "ellipse": ellipse,
                        "mask": fitted_mask,
                        "score": agreement,
                        "area": area,
                    })

    candidates.sort(
        key=lambda item: (item["score"], item["area"]),
        reverse=True,
    )

    selected = []

    for candidate in candidates:
        if any(
            binary_iou(candidate["mask"], previous["mask"]) > 0.5
            for previous in selected
        ):
            continue

        selected.append(candidate)

        if len(selected) >= config.max_candidates:
            break

    return selected


def region_grabcut_segment(rgb, config):
    height, width = rgb.shape[:2]
    scale = min(1.0, config.max_side / max(height, width))

    work = (
        cv2.resize(
            rgb,
            (
                max(1, round(width * scale)),
                max(1, round(height * scale)),
            ),
            interpolation=cv2.INTER_AREA,
        )
        if scale < 1 else rgb.copy()
    )

    accepted = []

    for proposal in region_proposals(work, config):
        mask = refine_proposal(work, proposal, config)

        if mask is None:
            continue

        if any(binary_iou(mask, previous) > 0.5 for previous in accepted):
            continue

        accepted.append(mask)

    labels = np.zeros(work.shape[:2], dtype=np.int32)
    next_id = 1

    for mask in accepted:
        available = mask & (labels == 0)
        components, count = ndimage.label(available)

        if count == 0:
            continue

        sizes = np.bincount(components.ravel())
        sizes[0] = 0
        largest = components == int(sizes.argmax())

        if largest.sum() < 0.5 * mask.sum():
            continue

        labels[largest] = next_id
        next_id += 1

    if scale < 1:
        labels = cv2.resize(
            labels,
            (width, height),
            interpolation=cv2.INTER_NEAREST,
        )

    return labels


region_rows = []

for index, record in enumerate(dev_records, 1):
    rgb = read_rgb(DATA_ROOT / record["path"])
    truth = load_mask(
        OUTPUT / "reference_masks" / f"{record['id']}.png",
        record["shape"],
    )

    started = time.perf_counter()
    prediction = region_grabcut_segment(rgb, region_config)
    elapsed = time.perf_counter() - started

    region_rows.append({
        "id": record["id"],
        "path": record["path"],
        "group": record["group"],
        "seconds": elapsed,
        **segmentation_metrics(truth, prediction),
    })

    save_mask(
        OUTPUT / "region_gc_dev_masks" / f"{record['id']}.png",
        prediction,
    )

    if index % 20 == 0:
        print(f"Processed {index}/{len(dev_records)} development images")

region_summary = aggregate(region_rows)

write_json(
    OUTPUT / "region_grabcut_experiment.json",
    {
        "name": "multithreshold_regions_grabcut",
        "scope": "development only",
        "config": asdict(region_config),
        "proposal_quantiles": [30, 45, 60, 75, 85],
        "minimum_region_ellipse_iou": 0.65,
        "provenance": reference_provenance,
        "dev": region_summary,
        "per_image": region_rows,
        "note": (
            "Follow-up experiment after initial evaluation was viewed. "
            "No evaluation images used for this experiment."
        ),
    },
)
write_csv(OUTPUT / "region_grabcut_dev.csv", region_rows)

print("\nDEVELOPMENT COMPARISON")
print(f"HSV baseline IoU: {best_experiment['dev']['iou']:.4f}")
print(f"Region + GrabCut IoU: {region_summary['iou']:.4f}")
print(f"Dice: {region_summary['dice']:.4f}")
print(f"Precision: {region_summary['precision']:.4f}")
print(f"Recall: {region_summary['recall']:.4f}")
print(f"Count MAE: {region_summary['count_error']:.4f}")
print(
    "Empty predictions:",
    sum(row["pred_instances"] == 0 for row in region_rows),
    "/", len(region_rows),
)
print("Baseline configuration and evaluation results were not changed.")

Processed 20/88 development images
Processed 40/88 development images
Processed 60/88 development images
Processed 80/88 development images

DEVELOPMENT COMPARISON
HSV baseline IoU: 0.3489
Region + GrabCut IoU: 0.4752
Dice: 0.5996
Precision: 0.5874
Recall: 0.6910
Count MAE: 5.8977
Empty predictions: 0 / 88
Baseline configuration and evaluation results were not changed.


## 12. Area-filter experiments


In [14]:
# NEW CELL — Development-only area-filter experiments.
# Reads cached region + GrabCut predictions; no GrabCut rerun.
# SAM is used for scoring only, never for selecting individual regions.

import numpy as np

verify_inputs()


def filter_instance_areas(labels, min_fraction, relative_fraction):
    """
    Keep instances whose area satisfies both:
      area / image_area >= min_fraction
      area / largest_predicted_instance >= relative_fraction

    Both thresholds use prediction geometry alone.
    Multiple instances can survive.
    """
    instance_ids, areas = np.unique(labels, return_counts=True)
    foreground = instance_ids > 0
    instance_ids = instance_ids[foreground]
    areas = areas[foreground]

    result = np.zeros(labels.shape, dtype=np.int32)

    if len(instance_ids) == 0:
        return result

    largest_area = int(areas.max())
    next_id = 1

    for instance_id, area in zip(instance_ids, areas):
        if (
            area / labels.size >= min_fraction
            and area / largest_area >= relative_fraction
        ):
            result[labels == instance_id] = next_id
            next_id += 1

    return result


# Declare variants before viewing results.
# Relative filters assume comparable object sizes and can miss smaller eggs.
area_trials = [
    ("unfiltered",       0.000, 0.00),
    ("absolute_005",     0.005, 0.00),
    ("absolute_010",     0.010, 0.00),
    ("absolute_020",     0.020, 0.00),
    ("absolute_030",     0.030, 0.00),
    ("relative_025",     0.000, 0.25),
    ("relative_050",     0.000, 0.50),
    ("combined_010_025", 0.010, 0.25),
    ("combined_020_025", 0.020, 0.25),
]

# Cache original predictions and references once.
cached_dev = []

for record in dev_records:
    prediction = load_mask(
        OUTPUT / "region_gc_dev_masks" / f"{record['id']}.png",
        record["shape"],
    )
    truth = load_mask(
        OUTPUT / "reference_masks" / f"{record['id']}.png",
        record["shape"],
    )
    cached_dev.append((record, prediction, truth))

# Bind the experiment to the exact cached inputs.
area_provenance = {
    "reference": reference_provenance,
    "prediction_sha256": {
        record["id"]: sha256_file(
            OUTPUT / "region_gc_dev_masks" / f"{record['id']}.png"
        )
        for record in dev_records
    },
}

area_experiments = []

for trial_name, minimum_area, relative_area in area_trials:
    rows = []

    for record, prediction, truth in cached_dev:
        filtered = filter_instance_areas(
            prediction,
            min_fraction=minimum_area,
            relative_fraction=relative_area,
        )

        rows.append({
            "id": record["id"],
            "path": record["path"],
            "group": record["group"],
            **segmentation_metrics(truth, filtered),
        })

    summary = aggregate(rows)
    empty_count = sum(row["pred_instances"] == 0 for row in rows)

    area_experiments.append({
        "name": trial_name,
        "min_area_fraction": minimum_area,
        "relative_area_fraction": relative_area,
        "dev": summary,
        "empty_predictions": empty_count,
        "per_image": rows,
    })

    print(
        f"{trial_name:20s} "
        f"IoU={summary['iou']:.4f} | "
        f"Dice={summary['dice']:.4f} | "
        f"P={summary['precision']:.4f} | "
        f"R={summary['recall']:.4f} | "
        f"Count MAE={summary['count_error']:.4f} | "
        f"Empty={empty_count}/{len(rows)}"
    )

# Keep the same primary selection objective: development macro IoU.
# Dice breaks ties; earlier trial order breaks remaining ties.
winner_index = max(
    range(len(area_experiments)),
    key=lambda index: (
        area_experiments[index]["dev"]["iou"],
        area_experiments[index]["dev"]["dice"],
        -index,
    ),
)

area_winner = area_experiments[winner_index]

for index, experiment in enumerate(area_experiments):
    experiment["decision"] = (
        "Selected among area-filter trials by development macro IoU, "
        "then Dice, then earlier trial order."
        if index == winner_index else
        "Discarded under the declared development selection rule."
    )

# Save selected candidate masks separately from baseline and eval masks.
for record, prediction, truth in cached_dev:
    filtered = filter_instance_areas(
        prediction,
        area_winner["min_area_fraction"],
        area_winner["relative_area_fraction"],
    )

    save_mask(
        OUTPUT / "region_gc_filtered_dev_masks" / f"{record['id']}.png",
        filtered,
    )

write_json(
    OUTPUT / "region_gc_area_experiments.json",
    {
        "scope": "development only",
        "selection": "macro IoU, then Dice, then earlier trial order",
        "provenance": area_provenance,
        "selected_trial": area_winner["name"],
        "trials": area_experiments,
        "limitations": (
            "Area filtering cannot recover eggs absent from the candidate "
            "masks. Relative filtering may discard smaller genuine eggs. "
            "Large background objects can survive."
        ),
        "evaluation_disclosure": (
            "Follow-up development after the initial evaluation was viewed. "
            "No evaluation images were scored in this cell."
        ),
    },
)

write_csv(
    OUTPUT / "region_gc_area_experiments.csv",
    [
        {
            "trial": experiment["name"],
            "min_area_fraction": experiment["min_area_fraction"],
            "relative_area_fraction": experiment["relative_area_fraction"],
            "dev_iou": experiment["dev"]["iou"],
            "dev_dice": experiment["dev"]["dice"],
            "dev_precision": experiment["dev"]["precision"],
            "dev_recall": experiment["dev"]["recall"],
            "dev_count_mae": experiment["dev"]["count_error"],
            "empty_predictions": experiment["empty_predictions"],
            "decision": experiment["decision"],
        }
        for experiment in area_experiments
    ],
)

print("\nSELECTED AREA-FILTER CANDIDATE")
print("Trial:", area_winner["name"])
print("Minimum image-area fraction:", area_winner["min_area_fraction"])
print("Minimum fraction of largest instance:", area_winner["relative_area_fraction"])

print("\nDEVELOPMENT COMPARISON")
print(
    f"HSV baseline:       "
    f"IoU={best_experiment['dev']['iou']:.4f}, "
    f"count MAE={best_experiment['dev']['count_error']:.4f}"
)
print(
    f"Unfiltered regions: "
    f"IoU={area_experiments[0]['dev']['iou']:.4f}, "
    f"count MAE={area_experiments[0]['dev']['count_error']:.4f}"
)
print(
    f"Selected filter:    "
    f"IoU={area_winner['dev']['iou']:.4f}, "
    f"count MAE={area_winner['dev']['count_error']:.4f}"
)
print("BEST_CONFIG and evaluation results remain unchanged.")

unfiltered           IoU=0.4752 | Dice=0.5996 | P=0.5874 | R=0.6910 | Count MAE=5.8977 | Empty=0/88
absolute_005         IoU=0.5094 | Dice=0.6254 | P=0.6399 | R=0.6875 | Count MAE=2.1477 | Empty=1/88
absolute_010         IoU=0.5317 | Dice=0.6413 | P=0.6732 | R=0.6871 | Count MAE=1.1705 | Empty=1/88
absolute_020         IoU=0.5496 | Dice=0.6516 | P=0.7106 | R=0.6815 | Count MAE=0.7500 | Empty=5/88
absolute_030         IoU=0.5535 | Dice=0.6474 | P=0.7218 | R=0.6614 | Count MAE=0.4773 | Empty=6/88
relative_025         IoU=0.5676 | Dice=0.6647 | P=0.7207 | R=0.6861 | Count MAE=0.8864 | Empty=0/88
relative_050         IoU=0.5461 | Dice=0.6358 | P=0.7260 | R=0.6246 | Count MAE=0.3977 | Empty=0/88
combined_010_025     IoU=0.5675 | Dice=0.6647 | P=0.7207 | R=0.6861 | Count MAE=0.6477 | Empty=1/88
combined_020_025     IoU=0.5657 | Dice=0.6623 | P=0.7296 | R=0.6815 | Count MAE=0.5795 | Empty=5/88

SELECTED AREA-FILTER CANDIDATE
Trial: relative_025
Minimum image-area fraction: 0.0
Minimum fractio

## 13. Freeze final image method and follow-up evaluation


In [15]:
# FREEZE AND EVALUATE THE NEW CLASSICAL PIPELINE
# Predictions use image evidence only:
# region proposals -> GrabCut -> relative area filtering.
# Original baseline files and BEST_CONFIG remain unchanged.

verify_inputs()

# Confirm cached development predictions have not changed since selection.
for record in dev_records:
    path = OUTPUT / "region_gc_dev_masks" / f"{record['id']}.png"
    expected = area_provenance["prediction_sha256"][record["id"]]

    if sha256_file(path) != expected:
        raise RuntimeError(
            "Cached development predictions changed. "
            "Rerun the area-filter experiments before freezing."
        )

if current_provenance() != area_provenance["reference"]:
    raise RuntimeError(
        "References changed since area-filter selection. "
        "Rerun development experiments."
    )

# Freeze independent copies of the selected settings.
FINAL_REGION_CONFIG = EllipseGrabCutConfig(**asdict(region_config))
FINAL_MIN_AREA = float(area_winner["min_area_fraction"])
FINAL_RELATIVE_AREA = float(area_winner["relative_area_fraction"])


def final_region_segment(rgb):
    instances = region_grabcut_segment(rgb, FINAL_REGION_CONFIG)

    return filter_instance_areas(
        instances,
        min_fraction=FINAL_MIN_AREA,
        relative_fraction=FINAL_RELATIVE_AREA,
    )


FOLLOWUP_OUTPUT = OUTPUT / "region_gc_final"
FOLLOWUP_OUTPUT.mkdir(parents=True, exist_ok=True)

final_definition = {
    "method": "multithreshold_regions_grabcut_area_filter",
    "region_config": asdict(FINAL_REGION_CONFIG),
    "proposal_quantiles": [30, 45, 60, 75, 85],
    "minimum_region_ellipse_iou": 0.65,
    "area_filter": {
        "selected_trial": area_winner["name"],
        "min_image_area_fraction": FINAL_MIN_AREA,
        "min_fraction_of_largest_instance": FINAL_RELATIVE_AREA,
    },
    "selection_metric": "development macro IoU",
    "development_results": area_winner["dev"],
    "provenance": current_provenance(),
    "protocol_note": (
        "Follow-up development occurred after viewing initial baseline "
        "evaluation. No evaluation scores were used by the follow-up "
        "candidate-selection cells. This is not a fresh untouched test."
    ),
}

write_json(
    FOLLOWUP_OUTPUT / "frozen_config.json",
    final_definition,
)

# Verify the live pipeline reproduces its cached development candidate
# on three images before evaluating.
for record in dev_records[:3]:
    rgb = read_rgb(DATA_ROOT / record["path"])
    live_prediction = final_region_segment(rgb)

    cached_prediction = load_mask(
        OUTPUT / "region_gc_filtered_dev_masks" / f"{record['id']}.png",
        record["shape"],
    )

    if not np.array_equal(live_prediction, cached_prediction):
        raise RuntimeError(
            f"Live/cached prediction mismatch: {record['path']}. "
            "Check whether proposal/refinement functions or settings changed. "
            "Evaluation stopped."
        )

print("Live pipeline matches cached development predictions on 3 checks.")

followup_rows = []

# Remove an old completion file so a failed rerun cannot appear complete.
(FOLLOWUP_OUTPUT / "evaluation.json").unlink(missing_ok=True)

for index, record in enumerate(eval_records, 1):
    rgb = read_rgb(DATA_ROOT / record["path"])
    truth = load_mask(
        OUTPUT / "reference_masks" / f"{record['id']}.png",
        record["shape"],
    )

    started = time.perf_counter()
    prediction = final_region_segment(rgb)
    elapsed = time.perf_counter() - started

    followup_rows.append({
        "id": record["id"],
        "path": record["path"],
        "group": record["group"],
        "seconds": elapsed,
        **segmentation_metrics(truth, prediction),
    })

    save_mask(
        FOLLOWUP_OUTPUT / "eval_masks" / f"{record['id']}.png",
        prediction,
    )

    print(f"Evaluated {index}/{len(eval_records)} images")

followup_summary = aggregate(followup_rows)

# Group bootstrap, consistent with the original evaluation.
grouped_ious = {}
for row in followup_rows:
    grouped_ious.setdefault(row["group"], []).append(row["iou"])

group_values = list(grouped_ious.values())
rng = np.random.default_rng(SEED)
bootstrap_means = []

for _ in range(2000):
    sampled_indices = rng.integers(
        0, len(group_values), size=len(group_values)
    )
    bootstrap_means.append(float(np.mean([
        value
        for index in sampled_indices
        for value in group_values[index]
    ])))

followup_summary["iou_group_bootstrap_95ci"] = [
    float(value)
    for value in np.percentile(bootstrap_means, [2.5, 97.5])
]
followup_summary["eval_groups"] = len(group_values)
followup_summary["empty_predictions"] = sum(
    row["pred_instances"] == 0
    for row in followup_rows
)

write_json(
    FOLLOWUP_OUTPUT / "evaluation.json",
    {
        "frozen_definition": final_definition,
        "summary": followup_summary,
        "images": followup_rows,
    },
)
write_csv(
    FOLLOWUP_OUTPUT / "evaluation.csv",
    followup_rows,
)

print("\nFOLLOW-UP EVALUATION — FROZEN CANDIDATE")
print(json.dumps(followup_summary, indent=2))
print("\nOriginal baseline and BEST_CONFIG were preserved.")

Live pipeline matches cached development predictions on 3 checks.
Evaluated 1/22 images
Evaluated 2/22 images
Evaluated 3/22 images
Evaluated 4/22 images
Evaluated 5/22 images
Evaluated 6/22 images
Evaluated 7/22 images
Evaluated 8/22 images
Evaluated 9/22 images
Evaluated 10/22 images
Evaluated 11/22 images
Evaluated 12/22 images
Evaluated 13/22 images
Evaluated 14/22 images
Evaluated 15/22 images
Evaluated 16/22 images
Evaluated 17/22 images
Evaluated 18/22 images
Evaluated 19/22 images
Evaluated 20/22 images
Evaluated 21/22 images
Evaluated 22/22 images

FOLLOW-UP EVALUATION — FROZEN CANDIDATE
{
  "iou": 0.5718647969659902,
  "dice": 0.667043846016859,
  "precision": 0.6669868273553766,
  "recall": 0.711823691332104,
  "boundary_f1": 0.4417680191917702,
  "count_error": 0.7272727272727273,
  "micro_iou": 0.5086125750763306,
  "images": 22,
  "iou_group_bootstrap_95ci": [
    0.4388396579060367,
    0.6976009123123815
  ],
  "eval_groups": 22,
  "empty_predictions": 0
}

Original bas

## 14. Final full-frame video detection and tracking


In [16]:
#
# Reuses VIDEO_PATH from your notebook.
# Example:
# VIDEO_PATH = Path("/kaggle/input/your-video-dataset/video.mp4")
#
# Classical video-specific detector:
# white regions -> shape validation -> local contrast/background validation.
#
# No fixed ROI, SAM, Grounding DINO, or GrabCut is used here.
# The image-dataset method and its evaluation remain unchanged.
#
# Output:
# /kaggle/working/egg_results/region_gc_final/video_tracking_full_frame/
#
# Limitations:
# - Intended for the supplied white-egg / dark-table video.
# - Hand-occluded and touching eggs can be missed.
# - Identity preservation through occlusion is not guaranteed.
# - Allocated track IDs are not a verified unique-egg count.

from pathlib import Path
import csv
import hashlib
import json
import time

import cv2
import numpy as np
from scipy.optimize import linear_sum_assignment
from IPython.display import display, FileLink


VIDEO_CONFIG = {
    "method": "full_frame_white_region_local_contrast",
    "minimum_brightness": 165,
    "max_saturation": 95,
    "min_image_area": 0.0015,
    "max_image_area": 0.045,
    "min_solidity": 0.80,
    "min_circularity": 0.45,
    "max_axis_ratio": 2.5,
    "min_ellipse_iou": 0.80,
    "minimum_local_contrast": 80,
    "maximum_interior_brightness_spread": 65,
    "ring_dark_threshold": 150,
    "minimum_dark_ring_fraction": 0.80,
}

TRACKER_CONFIG = {
    "max_missing_seconds": 0.5,
    "distance_gate": 2.0,
    "minimum_area_ratio": 0.25,
    "maximum_area_ratio": 4.0,
}

TRACKING_OUTPUT = Path(
    "/kaggle/working/egg_results/region_gc_final/"
    "video_tracking_full_frame"
)


def validate_configs():
    cfg = VIDEO_CONFIG

    if not 0 < cfg["min_image_area"] < cfg["max_image_area"] <= 1:
        raise ValueError("Invalid detection area thresholds.")

    for name in [
        "min_solidity",
        "min_circularity",
        "min_ellipse_iou",
        "minimum_dark_ring_fraction",
    ]:
        if not 0 <= cfg[name] <= 1:
            raise ValueError(f"{name} must be in [0, 1].")

    for name in [
        "minimum_brightness",
        "max_saturation",
        "ring_dark_threshold",
    ]:
        if not 0 <= cfg[name] <= 255:
            raise ValueError(f"{name} must be in [0, 255].")

    if cfg["max_axis_ratio"] < 1:
        raise ValueError("max_axis_ratio must be at least 1.")

    if (
        cfg["minimum_local_contrast"] < 0
        or cfg["maximum_interior_brightness_spread"] < 0
    ):
        raise ValueError("Contrast/spread thresholds cannot be negative.")

    if (
        TRACKER_CONFIG["max_missing_seconds"] < 0
        or TRACKER_CONFIG["distance_gate"] <= 0
        or not (
            0 < TRACKER_CONFIG["minimum_area_ratio"]
            <= TRACKER_CONFIG["maximum_area_ratio"]
        )
    ):
        raise ValueError("Invalid tracker configuration.")


def detect_video_eggs(rgb, config):
    if rgb.dtype != np.uint8 or rgb.ndim != 3 or rgb.shape[2] != 3:
        raise ValueError("Expected a uint8 RGB image.")

    height, width = rgb.shape[:2]

    smooth = cv2.GaussianBlur(rgb, (5, 5), 0)
    hsv = cv2.cvtColor(smooth, cv2.COLOR_RGB2HSV)
    brightness = hsv[..., 2]

    candidate = (
        (brightness >= config["minimum_brightness"])
        & (hsv[..., 1] <= config["max_saturation"])
    ).astype(np.uint8)

    kernel = cv2.getStructuringElement(
        cv2.MORPH_ELLIPSE, (3, 3)
    )

    candidate = cv2.morphologyEx(
        candidate, cv2.MORPH_OPEN, kernel
    )
    candidate = cv2.morphologyEx(
        candidate, cv2.MORPH_CLOSE, kernel
    )

    contours, _ = cv2.findContours(
        candidate,
        cv2.RETR_EXTERNAL,
        cv2.CHAIN_APPROX_NONE,
    )

    labels = np.zeros((height, width), dtype=np.int32)
    next_id = 1

    for contour in contours:
        if len(contour) < 5:
            continue

        area = cv2.contourArea(contour)
        area_fraction = area / (height * width)

        if not (
            config["min_image_area"]
            <= area_fraction
            <= config["max_image_area"]
        ):
            continue

        hull_area = cv2.contourArea(cv2.convexHull(contour))
        perimeter = cv2.arcLength(contour, True)

        solidity = area / max(hull_area, 1.0)
        circularity = (
            4 * np.pi * area / max(perimeter ** 2, 1.0)
        )

        if (
            solidity < config["min_solidity"]
            or circularity < config["min_circularity"]
        ):
            continue

        ellipse = cv2.fitEllipse(contour)
        (cx, cy), (axis_a, axis_b), angle = ellipse

        if (
            not np.isfinite([cx, cy, axis_a, axis_b, angle]).all()
            or min(axis_a, axis_b) <= 0
        ):
            continue

        axis_ratio = max(axis_a, axis_b) / min(axis_a, axis_b)

        if axis_ratio > config["max_axis_ratio"]:
            continue

        region = np.zeros((height, width), dtype=np.uint8)
        cv2.drawContours(
            region, [contour], -1, 1, cv2.FILLED
        )
        foreground = region > 0

        fitted = np.zeros((height, width), dtype=np.uint8)
        cv2.ellipse(fitted, ellipse, 1, -1)
        fitted = fitted > 0

        union = np.count_nonzero(foreground | fitted)
        ellipse_iou = (
            np.count_nonzero(foreground & fitted)
            / max(union, 1)
        )

        if ellipse_iou < config["min_ellipse_iou"]:
            continue

        # Immediate surroundings should be substantially darker than the egg.
        radius = max(3, round(min(axis_a, axis_b) * 0.20))
        ring_kernel = cv2.getStructuringElement(
            cv2.MORPH_ELLIPSE,
            (2 * radius + 1, 2 * radius + 1),
        )

        surrounding = (
            cv2.dilate(region, ring_kernel) > 0
        ) & ~foreground

        if np.count_nonzero(surrounding) < 10:
            continue

        local_contrast = (
            float(np.median(brightness[foreground]))
            - float(np.median(brightness[surrounding]))
        )

        interior_spread = float(
            np.percentile(brightness[foreground], 90)
            - np.percentile(brightness[foreground], 10)
        )

        dark_ring_fraction = float(np.mean(
            brightness[surrounding]
            < config["ring_dark_threshold"]
        ))

        if (
            local_contrast < config["minimum_local_contrast"]
            or interior_spread
            > config["maximum_interior_brightness_spread"]
            or dark_ring_fraction
            < config["minimum_dark_ring_fraction"]
        ):
            continue

        labels[foreground] = next_id
        next_id += 1

    return labels


def box_iou(a, b):
    left = np.maximum(a[:2], b[:2])
    right = np.minimum(a[2:], b[2:])

    intersection = float(np.prod(np.maximum(0, right - left)))
    area_a = float(np.prod(np.maximum(0, a[2:] - a[:2])))
    area_b = float(np.prod(np.maximum(0, b[2:] - b[:2])))

    return intersection / max(
        area_a + area_b - intersection,
        1e-9,
    )


class FullFrameEggTracker:
    def __init__(self, max_missed, config):
        if max_missed < 0:
            raise ValueError("max_missed cannot be negative.")

        self.max_missed = max_missed
        self.config = dict(config)
        self.tracks = []
        self.next_id = 1

    def update(self, boxes):
        boxes = [
            np.asarray(box, dtype=float)
            for box in boxes
        ]

        for box in boxes:
            if (
                box.shape != (4,)
                or not np.isfinite(box).all()
                or np.any(box[2:] <= box[:2])
            ):
                raise ValueError(
                    "Expected finite positive-area xyxy boxes."
                )

        # Predict translation only so box dimensions cannot become negative.
        for track in self.tracks:
            vx, vy = track["velocity"]
            track["box"] = track["box"] + np.array(
                [vx, vy, vx, vy]
            )
            track["missed"] += 1

        costs = np.full(
            (len(self.tracks), len(boxes)),
            1e6,
        )

        for i, track in enumerate(self.tracks):
            predicted = track["box"]
            center_a = (predicted[:2] + predicted[2:]) / 2

            scale = max(
                float(np.linalg.norm(predicted[2:] - predicted[:2])),
                1.0,
            )
            area_a = max(
                float(np.prod(predicted[2:] - predicted[:2])),
                1.0,
            )

            for j, box in enumerate(boxes):
                center_b = (box[:2] + box[2:]) / 2
                distance = (
                    float(np.linalg.norm(center_a - center_b))
                    / scale
                )
                area_ratio = (
                    float(np.prod(box[2:] - box[:2]))
                    / area_a
                )

                if (
                    distance <= self.config["distance_gate"]
                    and self.config["minimum_area_ratio"]
                    <= area_ratio
                    <= self.config["maximum_area_ratio"]
                ):
                    costs[i, j] = (
                        0.6 * (1 - box_iou(predicted, box))
                        + 0.4 * distance / self.config["distance_gate"]
                    )

        assignments = {}

        if costs.size:
            rows, columns = linear_sum_assignment(costs)

            for i, j in zip(rows, columns):
                if costs[i, j] >= 1e6:
                    continue

                track = self.tracks[i]
                old_center = (
                    track["last_observed"][:2]
                    + track["last_observed"][2:]
                ) / 2
                new_center = (boxes[j][:2] + boxes[j][2:]) / 2

                measured_velocity = (
                    new_center - old_center
                ) / track["missed"]

                track["velocity"] = (
                    0.6 * measured_velocity
                    + 0.4 * track["velocity"]
                )
                track["box"] = boxes[j].copy()
                track["last_observed"] = boxes[j].copy()
                track["missed"] = 0

                assignments[int(j)] = track["id"]

        self.tracks = [
            track for track in self.tracks
            if track["missed"] <= self.max_missed
        ]

        for index, box in enumerate(boxes):
            if index not in assignments:
                self.tracks.append({
                    "id": self.next_id,
                    "box": box.copy(),
                    "last_observed": box.copy(),
                    "velocity": np.zeros(2),
                    "missed": 0,
                })
                assignments[index] = self.next_id
                self.next_id += 1

        # Only observed detections receive displayed boxes.
        # Missing tracks stay internal; predictions are not reported as detections.
        return assignments


def instance_boxes(labels):
    boxes = []
    instance_ids = []

    for instance_id in np.unique(labels):
        if instance_id == 0:
            continue

        ys, xs = np.nonzero(labels == instance_id)

        boxes.append(np.array([
            xs.min(), ys.min(),
            xs.max() + 1, ys.max() + 1,
        ], dtype=float))
        instance_ids.append(int(instance_id))

    return boxes, instance_ids


def file_sha256(path):
    digest = hashlib.sha256()

    with Path(path).open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(block)

    return digest.hexdigest()


def run_full_frame_tracking(video_path):
    validate_configs()

    video_path = Path(video_path).resolve()

    if not video_path.is_file():
        raise FileNotFoundError(f"Video not found: {video_path}")

    TRACKING_OUTPUT.mkdir(parents=True, exist_ok=True)

    destination = TRACKING_OUTPUT / "tracked.mp4"
    csv_path = TRACKING_OUTPUT / "tracks.csv"
    summary_path = TRACKING_OUTPUT / "tracking_summary.json"

    if destination.resolve() == video_path:
        raise ValueError("Input and output video paths must differ.")

    summary_path.unlink(missing_ok=True)

    # Snapshot settings so the recorded configuration matches this run.
    detector_config = dict(VIDEO_CONFIG)
    tracker_config = dict(TRACKER_CONFIG)

    capture = cv2.VideoCapture(str(video_path))

    if not capture.isOpened():
        capture.release()
        raise RuntimeError(f"Cannot open video: {video_path}")

    fps = float(capture.get(cv2.CAP_PROP_FPS))
    width = int(capture.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(capture.get(cv2.CAP_PROP_FRAME_HEIGHT))
    expected_frames = int(capture.get(cv2.CAP_PROP_FRAME_COUNT))

    if (
        not np.isfinite(fps)
        or fps <= 0
        or min(width, height) < 1
    ):
        capture.release()
        raise RuntimeError("Invalid video metadata.")

    writer = cv2.VideoWriter(
        str(destination),
        cv2.VideoWriter_fourcc(*"mp4v"),
        fps,
        (width, height),
    )

    if not writer.isOpened():
        capture.release()
        writer.release()
        raise RuntimeError("MP4 encoder unavailable.")

    max_missed = max(
        0,
        round(fps * tracker_config["max_missing_seconds"]),
    )
    tracker = FullFrameEggTracker(
        max_missed=max_missed,
        config=tracker_config,
    )

    frame_index = 0
    detection_observations = 0
    frames_without_detections = 0
    processing_seconds = 0.0
    detection_histogram = {}
    run_start = time.perf_counter()

    try:
        with csv_path.open(
            "w", newline="", encoding="utf-8"
        ) as handle:
            csv_writer = csv.writer(handle)
            csv_writer.writerow([
                "frame", "time_seconds", "track_id",
                "x1", "y1", "x2", "y2",
            ])

            while True:
                success, frame = capture.read()

                if not success:
                    break

                if frame.shape[:2] != (height, width):
                    raise RuntimeError(
                        f"Unexpected dimensions at frame {frame_index}."
                    )

                frame_start = time.perf_counter()

                rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                labels = detect_video_eggs(rgb, detector_config)

                boxes, instance_ids = instance_boxes(labels)
                assignments = tracker.update(boxes)

                visible_count = len(boxes)
                detection_observations += visible_count
                frames_without_detections += int(visible_count == 0)
                detection_histogram[visible_count] = (
                    detection_histogram.get(visible_count, 0) + 1
                )

                for index, (box, instance_id) in enumerate(
                    zip(boxes, instance_ids)
                ):
                    track_id = assignments[index]

                    color = tuple(
                        int(value)
                        for value in np.random.default_rng(
                            track_id
                        ).integers(70, 255, 3)
                    )

                    region = labels == instance_id
                    frame[region] = (
                        0.80 * frame[region]
                        + 0.20 * np.array(color)
                    ).astype(np.uint8)

                    x1, y1, x2, y2 = map(int, box)

                    cv2.rectangle(
                        frame,
                        (x1, y1),
                        (x2 - 1, y2 - 1),
                        color,
                        2,
                    )
                    cv2.putText(
                        frame,
                        f"Egg {track_id}",
                        (x1, max(20, y1 - 6)),
                        cv2.FONT_HERSHEY_SIMPLEX,
                        0.6,
                        color,
                        2,
                    )

                    csv_writer.writerow([
                        frame_index,
                        frame_index / fps,
                        track_id,
                        x1, y1, x2, y2,
                    ])

                processing_seconds += (
                    time.perf_counter() - frame_start
                )

                cv2.putText(
                    frame,
                    (
                        f"Observed detections: {visible_count}"
                        f" | Frame: {frame_index}"
                    ),
                    (10, 25),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    0.6,
                    (0, 255, 255),
                    2,
                )

                writer.write(frame)
                frame_index += 1

                if frame_index % 300 == 0:
                    elapsed = time.perf_counter() - run_start
                    print(
                        f"Processed {frame_index}/"
                        f"{expected_frames if expected_frames > 0 else '?'}"
                        f" frames | "
                        f"{frame_index / max(elapsed, 1e-9):.2f} frames/s"
                    )

    finally:
        capture.release()
        writer.release()

    if frame_index == 0:
        raise RuntimeError("Input video has no decodable frames.")

    if expected_frames > 0 and frame_index < expected_frames:
        raise RuntimeError(
            f"Input decoding ended early: "
            f"{frame_index}/{expected_frames} frames."
        )

    # Fully decode the saved video to validate the persistent output.
    check = cv2.VideoCapture(str(destination))
    decoded_frames = 0

    try:
        if not check.isOpened():
            raise RuntimeError("Cannot reopen the output video.")

        while True:
            success, output_frame = check.read()

            if not success:
                break

            if output_frame.shape[:2] != (height, width):
                raise RuntimeError("Output frame dimensions are incorrect.")

            decoded_frames += 1

    finally:
        check.release()

    if decoded_frames != frame_index:
        raise RuntimeError(
            f"Output validation failed: "
            f"{decoded_frames}/{frame_index} frames decoded."
        )

    summary = {
        "method": detector_config["method"],
        "source_video": video_path.name,
        "source_sha256": file_sha256(video_path),
        "output_video": str(destination),
        "tracks_csv": str(csv_path),
        "frames": frame_index,
        "fps": fps,
        "width": width,
        "height": height,
        "duration_seconds": frame_index / fps,
        "processing_fps": (
            frame_index / max(processing_seconds, 1e-9)
        ),
        "wall_seconds_including_output_validation": (
            time.perf_counter() - run_start
        ),
        "allocated_track_ids": tracker.next_id - 1,
        "detection_observations": detection_observations,
        "frames_without_detections": frames_without_detections,
        "frames_by_detection_count": detection_histogram,
        "video_only_config": detector_config,
        "tracker_config": {
            **tracker_config,
            "max_missed_frames": max_missed,
        },
        "audio_preserved": False,
        "validation_note": (
            "Saved video was fully decoded and frame count checked. "
            "This validates encoding, not detection or identity accuracy."
        ),
        "limitations": (
            "Video-specific calibration for white eggs with dark local "
            "surroundings. Detector was inspected on sampled original-video "
            "frames, not scored against whole-video annotations. Hand "
            "occlusion, touching eggs, and bright rounded background objects "
            "can cause errors. Tracks may receive new IDs after extended "
            "occlusion or camera changes. Missing-track predictions are not "
            "displayed as confirmed detections. Allocated IDs are not a "
            "verified unique-egg count. Image-dataset evaluation scores "
            "do not apply to this video-only algorithm."
        ),
    }

    summary_path.write_text(
        json.dumps(summary, indent=2, allow_nan=False),
        encoding="utf-8",
    )

    print("\nTRACKING COMPLETE")
    print("Video:", destination)
    print("CSV:", csv_path)
    print("Summary:", summary_path)
    print("Validated output frames:", decoded_frames)
    print("Allocated track IDs:", tracker.next_id - 1)
    print(
        "Frames without detections:",
        f"{frames_without_detections}/{frame_index}",
        "(includes title screens and occlusions)",
    )

    return summary


# Tracker sanity checks.
validate_configs()

test_tracker = FullFrameEggTracker(
    max_missed=2,
    config=TRACKER_CONFIG,
)

assert test_tracker.update([[10, 10, 30, 40]])[0] == 1
assert test_tracker.update([[12, 10, 32, 40]])[0] == 1
assert test_tracker.update([]) == {}
assert test_tracker.update([[16, 10, 36, 40]])[0] == 1

expiry_tracker = FullFrameEggTracker(
    max_missed=1,
    config=TRACKER_CONFIG,
)

assert expiry_tracker.update([[10, 10, 30, 40]])[0] == 1
expiry_tracker.update([])
expiry_tracker.update([])
assert expiry_tracker.update([[10, 10, 30, 40]])[0] == 2

print("Tracker sanity checks passed.")

if "VIDEO_PATH" not in globals() or VIDEO_PATH is None:
    raise ValueError(
        "Set VIDEO_PATH to the original video path that worked previously."
    )

tracking_summary = None
tracking_summary = run_full_frame_tracking(VIDEO_PATH)

display(FileLink(tracking_summary["output_video"]))
display(FileLink(tracking_summary["tracks_csv"]))

Tracker sanity checks passed.
Processed 300/1440 frames | 118.42 frames/s
Processed 600/1440 frames | 79.12 frames/s
Processed 900/1440 frames | 73.71 frames/s
Processed 1200/1440 frames | 79.29 frames/s

TRACKING COMPLETE
Video: /kaggle/working/egg_results/region_gc_final/video_tracking_full_frame/tracked.mp4
CSV: /kaggle/working/egg_results/region_gc_final/video_tracking_full_frame/tracks.csv
Summary: /kaggle/working/egg_results/region_gc_final/video_tracking_full_frame/tracking_summary.json
Validated output frames: 1440
Allocated track IDs: 6
Frames without detections: 678/1440 (includes title screens and occlusions)


/kaggle/working/egg_results/region_gc_final/video_tracking_full_frame/tracked.mp4

/kaggle/working/egg_results/region_gc_final/video_tracking_full_frame/tracks.csv

## 15. Final report and submission export


In [17]:
# Final report and submission export. Rerun after updating REPORT_NOTES/Drive URL.
import importlib.metadata
import platform
import json
import zipfile
from urllib.parse import urlparse

verify_inputs()
if current_provenance() != final_definition["provenance"]:
    raise RuntimeError("References changed after freezing; rerun image experiments.")
if asdict(FINAL_REGION_CONFIG) != final_definition["region_config"]:
    raise RuntimeError("Final region configuration changed.")
if (
    FINAL_MIN_AREA != final_definition["area_filter"]["min_image_area_fraction"]
    or FINAL_RELATIVE_AREA
    != final_definition["area_filter"]["min_fraction_of_largest_instance"]
):
    raise RuntimeError("Final area filter changed.")

saved_evaluation = json.loads(
    (FOLLOWUP_OUTPUT / "evaluation.json").read_text(encoding="utf-8")
)
if saved_evaluation["frozen_definition"] != final_definition:
    raise RuntimeError("Saved evaluation belongs to a different final definition.")
final_summary = saved_evaluation["summary"]
final_rows = saved_evaluation["images"]

if tracking_summary is not None:
    if tracking_summary["video_only_config"] != VIDEO_CONFIG:
        raise RuntimeError("Video settings changed since tracking; rerun video cell.")
    if tracking_summary["tracker_config"] != {
        **TRACKER_CONFIG,
        "max_missed_frames": round(
            tracking_summary["fps"] * TRACKER_CONFIG["max_missing_seconds"]
        ),
    }:
        raise RuntimeError("Tracker settings changed; rerun video cell.")
    if not Path(tracking_summary["output_video"]).is_file():
        raise FileNotFoundError("Final tracked video is missing.")

if VIDEO_OUTPUT_LINK:
    parsed = urlparse(VIDEO_OUTPUT_LINK)
    if parsed.scheme != "https" or parsed.hostname != "drive.google.com":
        raise ValueError("Use an HTTPS Google Drive output-video link.")


def note(name):
    return REPORT_NOTES.get(name, "").strip() or "NOT RECORDED."


def final_comparison(row):
    record = next(r for r in records if r["id"] == row["id"])
    rgb = read_rgb(DATA_ROOT / record["path"])
    truth = load_mask(
        OUTPUT / "reference_masks" / f"{record['id']}.png", record["shape"]
    )
    prediction = load_mask(
        FOLLOWUP_OUTPUT / "eval_masks" / f"{record['id']}.png", record["shape"]
    )
    errors = (rgb * 0.35).astype(np.uint8)
    errors[(truth > 0) & (prediction > 0)] = (0, 200, 0)
    errors[(truth == 0) & (prediction > 0)] = (255, 0, 0)
    errors[(truth > 0) & (prediction == 0)] = (255, 0, 255)
    figure, axes = plt.subplots(1, 4, figsize=(14, 4))
    for axis, image, title in zip(
        axes, [rgb, truth, prediction, errors],
        ["Original", "SAM reference", "Final classical instances",
         "Green TP / red FP / magenta FN"],
    ):
        if image.ndim == 2:
            axis.imshow(image, cmap="nipy_spectral", interpolation="nearest")
        else:
            axis.imshow(image)
        axis.set_title(title)
        axis.axis("off")
    figure.suptitle(
        f"{row['path']} | IoU={row['iou']:.3f}, P={row['precision']:.3f}, "
        f"R={row['recall']:.3f} | count "
        f"{row['truth_instances']} -> {row['pred_instances']}", fontsize=9
    )
    figure.tight_layout()
    return figure


# Retain every executed experiment family, including unsuccessful trials.
trial_log = []
for family, collection in [
    ("Initial classical baselines", experiments),
    ("Contour ellipse + GrabCut", new_experiments),
]:
    for item in collection:
        trial_log.append({
            "family": family, "name": item["name"], "config": item["config"],
            "rationale": item["rationale"], "dev": item["dev"],
            "decision": item["decision"], "mean_seconds": item["mean_seconds"],
        })
trial_log.append({
    "family": "Multi-threshold region + GrabCut",
    "name": "multithreshold_regions_grabcut",
    "config": {
        **asdict(region_config), "quantiles": [30, 45, 60, 75, 85],
        "minimum_region_ellipse_iou": 0.65,
    },
    "rationale": "Replace incomplete edge-contour proposals with color/intensity regions.",
    "dev": region_summary,
    "decision": "Retained as proposal/refinement core; unfiltered output over-counted objects.",
    "mean_seconds": float(np.mean([r["seconds"] for r in region_rows])),
})
for item in area_experiments:
    trial_log.append({
        "family": "Area-filter ablations", "name": item["name"],
        "config": {
            "min_area_fraction": item["min_area_fraction"],
            "relative_area_fraction": item["relative_area_fraction"],
        },
        "rationale": "Reject small spurious instances using predicted areas alone.",
        "dev": item["dev"], "decision": item["decision"],
        "mean_seconds": None,
    })
write_json(OUTPUT / "final_experiment_log.json", trial_log)
write_csv(OUTPUT / "final_experiment_log.csv", [
    {
        "family": t["family"], "trial": t["name"],
        "parameters": json.dumps(t["config"], sort_keys=True),
        "rationale": t["rationale"], "dev_iou": t["dev"]["iou"],
        "dev_dice": t["dev"]["dice"], "dev_precision": t["dev"]["precision"],
        "dev_recall": t["dev"]["recall"], "dev_count_mae": t["dev"]["count_error"],
        "decision": t["decision"],
    }
    for t in trial_log
])

report_path = OUTPUT / "report.pdf"
with PdfPages(report_path) as pdf:
    report_text_page(pdf, "Egg segmentation and individual tracking", [
        "Dataset: Fire / egg, Roboflow egg-vzhtr, version 1.\n"
        "https://universe.roboflow.com/fire-bf7qe/egg-vzhtr/dataset/1\n"
        "License displayed on Universe: CC BY 4.0. Credit the creator.",
        f"Images: {len(records)}; development: {len(dev_records)}; "
        f"evaluation: {len(eval_records)}; source groups: {len(groups)}.",
        "Task 1: Grounding DINO locates eggs from 'an egg.'. Pretrained SAM "
        "predicts three candidates per box; highest predicted mask IoU selects "
        "the candidate. Higher-confidence masks own overlap; near duplicates "
        "are suppressed. Integer PNG masks preserve instances.",
        f"Detector: {DETECTOR_NAME}\nSAM: {SAM_NAME}\n"
        f"Detection thresholds: box {BOX_THRESHOLD}, text {TEXT_THRESHOLD}.",
        "Reference review: " + note("reference_review"),
        "SAM masks are pseudo-ground-truth, not independent human annotations. "
        "Reported segmentation scores measure agreement with SAM.",
        "AI disclosure: " + note("ai_disclosure"),
        "Status: " + (
            "Video generated and Drive link supplied; verify public access and finish notes."
            if tracking_summary and VIDEO_OUTPUT_LINK else
            "DRAFT: tracking or public Drive link incomplete."
        ),
    ])
    report_text_page(pdf, "Protocol and final image method", [
        f"Split seed: {SEED}; pHash Hamming threshold: {PHASH_DISTANCE}; "
        f"eval fraction: {len(eval_records)/len(records):.3f}.",
        "Leakage reduction groups exact duplicates, similar pHashes including "
        "flips, matching Roboflow source stems, and optional manual source "
        "groups. Subset sum chooses the nearest 20% image count without "
        "splitting groups. Hashes cannot guarantee capture-session independence.",
        "Source-group review: " + note("source_group_review"),
        "Initial baseline trials used development macro IoU, then Dice, then "
        "earlier trial order. Subsequent development occurred after the first "
        "evaluation was viewed. Follow-up candidate cells used development "
        "scores only, but the follow-up evaluation is not a fresh untouched test.",
        "Final method: threshold Lab lightness/chroma, HSV value and inverse "
        "saturation at fixed quantiles [30,45,60,75,85], both polarities. "
        "Open/close regions, reject unsuitable contours, fit ellipses, require "
        "region/ellipse agreement >=0.65, and suppress duplicate proposals. "
        "Local GrabCut uses ellipse-derived seeds. Validate refined components, "
        "suppress duplicates, and reject instances small relative to the largest.",
        "No SAM boxes, masks or other pretrained outputs generate classical "
        "predictions. SAM is used only for development/evaluation scoring.",
        "Region parameters: " + json.dumps(final_definition["region_config"], sort_keys=True),
        "Final area filter: " + json.dumps(final_definition["area_filter"], sort_keys=True),
    ])
    report_text_page(pdf, "Final follow-up evaluation", [
        f"Macro IoU: {final_summary['iou']:.4f}\nDice: {final_summary['dice']:.4f}\n"
        f"Precision: {final_summary['precision']:.4f}\nRecall: {final_summary['recall']:.4f}\n"
        f"Boundary F1 (2 pixels): {final_summary['boundary_f1']:.4f}\n"
        f"Micro IoU: {final_summary['micro_iou']:.4f}\n"
        f"Count MAE: {final_summary['count_error']:.4f}\n"
        f"Empty predictions: {final_summary['empty_predictions']}/{final_summary['images']}",
        "95% group-bootstrap IoU interval (2000 draws): "
        + str(final_summary["iou_group_bootstrap_95ci"])
        + f"; eval groups: {final_summary['eval_groups']}. Small groups make uncertainty unstable.",
        f"Original baseline eval IoU: {evaluation_summary['iou']:.4f}; "
        f"baseline Dice: {evaluation_summary['dice']:.4f}; "
        f"baseline count MAE: {evaluation_summary['count_error']:.4f}. "
        "Baseline results remain separate from final results.",
        "IoU=TP/(TP+FP+FN); Dice=2TP/(2TP+FP+FN). Precision=TP/(TP+FP), "
        "recall=TP/(TP+FN). Boundary F1 matches boundaries within 2 pixels. "
        "Macro metrics weight images equally; micro IoU pools pixels. "
        "Count MAE is not instance AP or correspondence accuracy.",
        "Two empty masks score IoU/Dice/boundary F1=1; a single empty mask "
        "scores these=0. Empty prediction against nonempty truth has "
        "precision=0; empty truth has recall=1.",
        "Agreement is moderate, not perfect. Boundary error, missed egg pixels "
        "and accepted background remain. Relative filtering can lose smaller "
        "eggs; proposal rejection can miss clipped or obscured eggs.",
        "Applicant qualitative analysis: " + note("qualitative_analysis"),
    ])
    figure, axes = plt.subplots(1, 3, figsize=(14, 4))
    axes[0].hist([r["iou"] for r in final_rows], bins=np.linspace(0, 1, 11))
    axes[0].set(xlabel="Per-image IoU", ylabel="Images", title="Final eval distribution")
    axes[1].scatter([r["truth_fraction"] for r in final_rows], [r["iou"] for r in final_rows])
    axes[1].set(xlabel="SAM foreground fraction", ylabel="IoU", ylim=(0, 1),
                title="Scale vs agreement")
    axes[2].bar(["FP", "FN"], [sum(r["fp"] for r in final_rows),
                              sum(r["fn"] for r in final_rows)], color=["red", "magenta"])
    axes[2].set(ylabel="Pixels", title="Leakage vs omission")
    figure.tight_layout()
    pdf.savefig(figure)
    figure.savefig(OUTPUT / "final_evaluation_plots.png", dpi=150)
    plt.close(figure)
    for t in trial_log:
        report_text_page(pdf, f"Experiment: {t['name']}", [
            "Family: " + t["family"],
            "Change / motivation: " + t["rationale"],
            "Parameters: " + json.dumps(t["config"], sort_keys=True),
            "Development results: " + json.dumps(t["dev"], sort_keys=True),
            "Decision: " + t["decision"],
            "Mean segmentation seconds/image: " + (
                f"{t['mean_seconds']:.4f}" if t["mean_seconds"] is not None
                else "Not measured for cached-mask filter."
            ),
            "Applicant observations: " + REPORT_NOTES.get("experiment_notes", {}).get(
                t["name"], "No additional manual observations recorded."
            ),
        ])
    report_text_page(pdf, "Diagnostic ablations and improvements", [
        "Watershed development diagnostic: " + json.dumps({
            name: aggregate(rows) for name, rows in diagnostic_rows.items()
        }, sort_keys=True),
        "Removing watershed reduced count error with almost no IoU change. "
        "HSV stage plots showed high egg-pixel recall but low precision before "
        "whole-component filtering removed the regions containing eggs.",
        "Contour ellipse proposals selected background or missed incomplete "
        "egg boundaries. Multi-threshold regions improved recall but produced "
        "too many objects. Area filtering improved precision and count error.",
        "Video experiment history: strict ellipse filtering removed a real egg "
        "while leaving background. Pale/bright validation retained shirt patches. "
        "Fixed table ROI cut off eggs. Final video detector instead uses full-frame "
        "color/shape plus dark-surrounding contrast, without a fixed ROI.",
        "Future improvements: better proposal completeness for dirty/clipped "
        "eggs, handling mixed instance sizes, annotated independent test data, "
        "and occlusion/camera-transition-aware identity association.",
    ])
    ordered = sorted(final_rows, key=lambda r: r["iou"])
    chosen = ordered[:4] + [r for r in ordered[-2:] if r not in ordered[:4]]
    for rank, row in enumerate(chosen, 1):
        figure = final_comparison(row)
        pdf.savefig(figure)
        figure.savefig(OUTPUT / f"final_qualitative_{rank}.png", dpi=150)
        plt.close(figure)
    report_text_page(pdf, "Final video method and tracking", [
        "Assigned video: https://www.youtube.com/watch?v=Y4JyV94y2_A",
        "Video-only method: full-frame bright low-saturation regions, morphology, "
        "area/solidity/circularity/ellipse checks, high local contrast, low "
        "interior brightness spread and a mostly dark surrounding ring. "
        "No ROI or pretrained model is used for video detection.",
        "Parameters: " + json.dumps(VIDEO_CONFIG, sort_keys=True),
        "Tracker: translation-only constant-velocity prediction; Hungarian "
        "assignment combines center distance and IoU, with distance/area gates. "
        "Unmatched detections get new IDs. Missing tracks survive a limited "
        "interval internally; predicted positions are not confirmed detections.",
        "Run: " + (json.dumps({
            key: tracking_summary[key] for key in [
                "frames", "fps", "allocated_track_ids",
                "frames_without_detections", "processing_fps",
            ]
        }) if tracking_summary else "NOT RUN."),
        "Review: " + note("video_review"),
        "No measured precision, IDF1 or MOTA is claimed without annotated video. "
        "Hand occlusion, touching eggs and camera changes can cause misses or "
        "new IDs. Title screens also contribute to frames without detections. "
        "The image dataset scores do not evaluate this separately calibrated "
        "video detector. Output does not preserve original audio.",
        "Public Google Drive video:\n" + (VIDEO_OUTPUT_LINK or "MISSING"),
    ])
    if VIDEO_OUTPUT_LINK:
        figure = plt.figure(figsize=(8.27, 11.69))
        figure.text(0.08, 0.9, "Open final tracking video", fontsize=16, url=VIDEO_OUTPUT_LINK)
        figure.text(0.08, 0.83, "\n".join(textwrap.wrap(VIDEO_OUTPUT_LINK, 80)), fontsize=9)
        pdf.savefig(figure)
        plt.close(figure)

environment = {
    "python": platform.python_version(), "device": DEVICE, "packages": {}
}
for package in [
    "numpy", "opencv-python-headless", "Pillow", "scipy", "scikit-image",
    "matplotlib", "torch", "torchvision", "transformers", "gdown",
]:
    environment["packages"][package] = importlib.metadata.version(package)
write_json(OUTPUT / "environment.json", environment)
(OUTPUT / "requirements.txt").write_text(
    "numpy>=1.26,<3\nopencv-python-headless>=4.10,<4.12\nPillow>=10.4,<13\n"
    "scipy>=1.13,<2\nscikit-image>=0.24,<1\nmatplotlib>=3.9,<4\n"
    "torch>=2.5,<3\ntorchvision>=0.20,<1\ntransformers==4.49.0\ngdown>=5,<6\n",
    encoding="utf-8",
)
(OUTPUT / "requirements-exact.txt").write_text(
    "\n".join(f"{k}=={v}" for k, v in environment["packages"].items()) + "\n",
    encoding="utf-8",
)
(OUTPUT / "README.md").write_text(
    "# Egg segmentation and tracking\n\n"
    "Open final.ipynb in Kaggle; enable Internet and a GPU. Edit Configuration "
    "with the extracted dataset location (or use the assignment ZIP download) "
    "and the original uploaded video path. Run in order. Inspect the reference "
    "audit and source groups; then explicitly complete the review confirmation "
    "cell before proceeding. Retain baseline and unsuccessful experiments. "
    "The image method is region/GrabCut/area filtering; the final video method "
    "is separately calibrated full-frame color/shape/local contrast.\n\n"
    "Set REPORT_NOTES to your actual observations. Upload the final tracked.mp4 "
    "to Google Drive with Anyone-with-link sharing, set VIDEO_OUTPUT_LINK, and "
    "rerun the report cell. Download report.pdf, logs, requirements and this "
    "notebook for the public GitHub repository. Dataset attribution: "
    "Fire / egg-vzhtr version 1, CC BY 4.0 as displayed on Universe.\n\n"
    "Scores measure agreement with SAM. Follow-up development occurred after "
    "the initial eval was viewed; do not present it as a fresh untouched test. "
    "No annotated video accuracy is claimed. Never publish API keys or "
    "unlicensed media.\n", encoding="utf-8",
)

export_path = Path("/kaggle/working/egg_submission_outputs.zip")
# Export requested deliverables and experiment records, not obsolete videos.
allowed_suffixes = {".json", ".csv", ".pdf", ".png", ".txt", ".md"}
current_video = Path(tracking_summary["output_video"]) if tracking_summary else None
with zipfile.ZipFile(export_path, "w", zipfile.ZIP_DEFLATED) as archive:
    for path in sorted(OUTPUT.rglob("*")):
        if not path.is_file():
            continue
        # Exclude superseded tracking directories from the submission bundle.
        relative = path.relative_to(OUTPUT)
        if any(part.startswith("video_tracking") for part in relative.parts):
            if current_video is None or path.parent != current_video.parent:
                continue
        if path.suffix.lower() in allowed_suffixes or path == current_video:
            archive.write(path, arcname=relative)

print("Final report:", report_path)
print("Submission outputs:", export_path)
print("Download/save final.ipynb separately and include it in the repository.")
if not tracking_summary or not VIDEO_OUTPUT_LINK:
    print("DRAFT: final video or public Google Drive link is missing.")
missing_notes = [
    name for name in ["reference_review", "source_group_review",
                     "qualitative_analysis", "video_review"]
    if not REPORT_NOTES.get(name, "").strip()
]
if missing_notes:
    print("Written observations still required:", ", ".join(missing_notes))
display(FileLink(str(report_path)))
display(FileLink(str(export_path)))


Final report: /kaggle/working/egg_results/report.pdf
Submission outputs: /kaggle/working/egg_submission_outputs.zip
Download/save final.ipynb separately and include it in the repository.
DRAFT: final video or public Google Drive link is missing.
Written observations still required: reference_review, source_group_review, qualitative_analysis


/kaggle/working/egg_results/report.pdf

/kaggle/working/egg_submission_outputs.zip